In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import time

# ---------------------------------------------------------
# 1. Helper Function: Newton-Schulz Matrix Inversion
# ---------------------------------------------------------
def batched_newton_schulz_inverse(A, num_iters=6):
    """
    Computes the inverse of a batch of matrices using Newton-Schulz iteration.
    This is highly stable for backpropagation as it avoids torch.linalg.inv
    and relies entirely on pure matrix multiplications.
    """
    B, N, _ = A.shape
    I = torch.eye(N, device=A.device).unsqueeze(0).expand(B, N, N)
    
    # 1. Guaranteed Convergent Initialization (Alpha * A^T)
    frob_norm_sq = torch.sum(A * A, dim=(-2, -1), keepdim=True)
    X = A.transpose(-2, -1) / (frob_norm_sq + 1e-6)
    
    # 2. Iterative Refinement: X_{k+1} = X_k (2I - A X_k)
    for _ in range(num_iters):
        AX = torch.bmm(A, X)
        X = torch.bmm(X, (2.0 * I) - AX)
        
    return X

# ---------------------------------------------------------
# 2. CLOUDS Model (Exact Stan-Logic + PyTorch GEM)
# ---------------------------------------------------------
class CLOUDS(nn.Module):
    def __init__(self, obs_dim, latent_dim, covar_dim, delta=1e-4, theta_mode="exact"):
        super().__init__()
        self.D = obs_dim
        self.K = latent_dim
        self.C_dim = covar_dim
        self.delta = delta
        self.theta_mode = theta_mode
        
        # 1. Continuous-Time Dynamics (Stan Parameterization)
        if self.theta_mode == "exact":
            # S: Symmetric Positive Definite Component
            self.L_S = nn.Parameter(torch.eye(self.K) + 0.1 * torch.randn(self.K, self.K))
            # A: Skew-Symmetric Rotational Component
            self.gamma_skew = nn.Parameter(torch.randn(self.K, self.K) * 0.1)
            # Omega: Stationary Covariance
            self.L_Omega = nn.Parameter(torch.eye(self.K) + 0.1 * torch.randn(self.K, self.K))
        elif self.theta_mode == "diagonal":
            self.log_rho = nn.Parameter(torch.randn(self.K) * 0.1 - 2.0)
            
        # 2. Linear Mean Function
        self.B = nn.Parameter(torch.randn(self.K, self.C_dim) * 0.1)
        self.C_int = nn.Parameter(torch.randn(self.K, self.C_dim) * 0.1)
        self.d_bias = nn.Parameter(torch.randn(self.K) * 0.1)
        
        # 3. Factor Loadings (Structural PCA identifiable)
        self.Z = nn.Parameter(torch.randn(self.D, self.K) - 0.5) 
        self.register_buffer('tril_mask', torch.tril(torch.ones(self.D, self.K)))
        
        # 4. Heteroscedastic Observation Noise
        self.log_psi = nn.Parameter(torch.zeros(self.D)) 

    def get_dynamics(self):
        device = self.Z.device
        if self.theta_mode == "exact":
            # Build S (Symmetric PD)
            L_S_tril = torch.tril(self.L_S)
            S = L_S_tril @ L_S_tril.T + self.delta * torch.eye(self.K, device=device)
            
            # Build A (Skew-Symmetric)
            A_skew = self.gamma_skew - self.gamma_skew.T
            
            # Build Omega (Stationary Covariance)
            L_Omega_tril = torch.tril(self.L_Omega)
            Omega = L_Omega_tril @ L_Omega_tril.T + self.delta * torch.eye(self.K, device=device)
            
            # Safe Inverse of Omega using Newton-Schulz
            Omega_inv = batched_newton_schulz_inverse(Omega.unsqueeze(0), num_iters=6).squeeze(0)
            
            # Exact Drift Matrix mapping: Theta = (S + A) * Omega^-1
            Theta = (S + A_skew) @ Omega_inv
            return Theta, Omega
            
        # Exact Diagonal Fallback
        Theta = torch.diag(torch.exp(self.log_rho))
        Omega = torch.eye(self.K, device=device) # Unused in diagonal analytic solution
        return Theta, Omega

    def get_subject_matrices(self, Theta, Omega, u, times):
        dt = times[1:] - times[:-1]
        device = self.Z.device
        
        Theta_batch = Theta.unsqueeze(0).expand(dt.shape[0], self.K, self.K)
        A_trans = torch.linalg.matrix_exp(-Theta_batch * dt.view(-1, 1, 1))
        
        u_t, t_val = u[1:], times[1:].unsqueeze(1)
        mu = u_t @ self.B.T + (u_t * t_val) @ self.C_int.T + self.d_bias
        
        I_batch = torch.eye(self.K, device=device).unsqueeze(0).expand(dt.shape[0], self.K, self.K)
        b_shift = torch.bmm(I_batch - A_trans, mu.unsqueeze(-1)).squeeze(-1)
        
        # EXACT Continuous Transition Covariance (Q)
        if self.theta_mode == "diagonal":
            theta_diag = torch.diag(Theta)
            Q_diag = (1.0 - torch.exp(-2.0 * theta_diag.unsqueeze(0) * dt.unsqueeze(1))) / (2.0 * theta_diag.unsqueeze(0))
            Q = torch.diag_embed(Q_diag)
        else:
            Omega_batch = Omega.unsqueeze(0).expand(dt.shape[0], self.K, self.K)
            A_trans_T = A_trans.transpose(1, 2)
            # Q = Omega - Phi * Omega * Phi^T
            Q = Omega_batch - torch.bmm(A_trans, torch.bmm(Omega_batch, A_trans_T))
            # Force absolute mathematical symmetry
            Q = 0.5 * (Q + Q.transpose(1, 2))
            
        Lambda = self.tril_mask * torch.exp(self.Z)
        return A_trans, b_shift, dt, Lambda, Q

    def kalman_smoother(self, x_obs, A_trans, b_shift, dt, Lambda, Q):
        T = x_obs.shape[0]
        device = x_obs.device
        
        f_pred, P_pred = torch.zeros(T, self.K, device=device), torch.zeros(T, self.K, self.K, device=device)
        f_filt, P_filt = torch.zeros(T, self.K, device=device), torch.zeros(T, self.K, self.K, device=device)
        f_filt[0], P_filt[0] = torch.zeros(self.K, device=device), torch.eye(self.K, device=device)
        
        # Heteroscedastic Noise matrix
        R_mat = torch.diag(torch.exp(self.log_psi))
        I_k = torch.eye(self.K, device=device)
        
        for j in range(1, T):
            idx = j - 1
            f_pred[j] = A_trans[idx] @ f_filt[j-1] + b_shift[idx]
            # Exact Q added here instead of dt * I
            P_pred[j] = A_trans[idx] @ P_filt[j-1] @ A_trans[idx].T + Q[idx]
            
            if torch.isnan(x_obs[j]).all():
                f_filt[j], P_filt[j] = f_pred[j], P_pred[j]
            else:
                x_pred = Lambda @ f_pred[j]
                S_t = Lambda @ P_pred[j] @ Lambda.T + R_mat
                K_gain = P_pred[j] @ Lambda.T @ torch.linalg.inv(S_t)
                f_filt[j] = f_pred[j] + K_gain @ (x_obs[j] - x_pred)
                P_filt[j] = (I_k - K_gain @ Lambda) @ P_pred[j]
            
        f_smooth, P_smooth, P_cross = torch.zeros_like(f_filt), torch.zeros_like(P_filt), torch.zeros_like(P_filt)
        f_smooth[-1], P_smooth[-1] = f_filt[-1], P_filt[-1]
        
        for j in range(T-2, -1, -1):
            J_t = P_filt[j] @ A_trans[j].T @ torch.linalg.inv(P_pred[j+1])
            f_smooth[j] = f_filt[j] + J_t @ (f_smooth[j+1] - f_pred[j+1])
            P_smooth[j] = P_filt[j] + J_t @ (P_smooth[j+1] - P_pred[j+1]) @ J_t.T
            P_cross[j+1] = J_t @ P_smooth[j+1]
            
        return f_smooth, P_smooth, P_cross

    def expected_complete_log_posterior_vectorized(self, subjects_data, smoothed_stats, Theta, Omega, Lambda):
        ll_obs, ll_lat = 0.0, 0.0
        
        inv_psi = torch.exp(-self.log_psi)
        L_Psi_L = Lambda.T @ torch.diag(inv_psi) @ Lambda 
        
        for i, subj in enumerate(subjects_data):
            x_obs, u, times = subj['x'], subj['u'], subj['t']
            f_s, P_s, P_c = smoothed_stats[i]
            
            A_trans, b_shift, dt, _, Q_exact = self.get_subject_matrices(Theta, Omega, u, times)
            
            # --- 1. Observation Likelihood (Heteroscedastic) ---
            valid_mask = ~torch.isnan(x_obs).any(dim=1)
            if valid_mask.any():
                x_v, f_v, P_v = x_obs[valid_mask], f_s[valid_mask], P_s[valid_mask]
                trace_E = torch.sum(P_v * L_Psi_L.unsqueeze(0), dim=(1,2)) + torch.sum(f_v * (f_v @ L_Psi_L), dim=1)
                term_obs = torch.sum((x_v**2) * inv_psi, dim=1) - 2 * torch.sum(x_v * (f_v @ Lambda.T) * inv_psi, dim=1) + trace_E
                ll_obs += torch.sum(-0.5 * term_obs - 0.5 * torch.sum(self.log_psi))
                
            # --- 2. Latent Likelihood (Exact Continuous Trace) ---
            Q_stable = Q_exact + 1e-5 * torch.eye(self.K, device=Q_exact.device).unsqueeze(0)
            Q_inv = torch.linalg.inv(Q_stable)
            log_det_Q = torch.linalg.slogdet(Q_stable)[1]
            
            f_j, f_jm1 = f_s[1:], f_s[:-1]
            P_j, P_jm1, P_cj = P_s[1:], P_s[:-1], P_c[1:]
            
            # Sub-matrices of the quadratic expectation
            E_jj = P_j + torch.bmm(f_j.unsqueeze(-1), f_j.unsqueeze(1))
            E_jjm1 = P_cj + torch.bmm(f_j.unsqueeze(-1), f_jm1.unsqueeze(1))
            E_jm1jm1 = P_jm1 + torch.bmm(f_jm1.unsqueeze(-1), f_jm1.unsqueeze(1))
            
            A_T = A_trans.transpose(1, 2)
            Term1 = E_jj
            Term2 = - torch.bmm(E_jjm1, A_T)
            Term3 = - torch.bmm(A_trans, E_jjm1.transpose(1, 2))
            Term4 = torch.bmm(A_trans, torch.bmm(E_jm1jm1, A_T))
            
            b_uns, b_T = b_shift.unsqueeze(-1), b_shift.unsqueeze(1)
            f_j_uns, f_j_T = f_j.unsqueeze(-1), f_j.unsqueeze(1)
            f_jm1_uns, f_jm1_T = f_jm1.unsqueeze(-1), f_jm1.unsqueeze(1)
            
            Term5 = - torch.bmm(f_j_uns, b_T)
            Term6 = - torch.bmm(b_uns, f_j_T)
            Term7 = torch.bmm(A_trans, torch.bmm(f_jm1_uns, b_T))
            Term8 = torch.bmm(b_uns, torch.bmm(f_jm1_T, A_T))
            Term9 = torch.bmm(b_uns, b_T)
            
            M_j = Term1 + Term2 + Term3 + Term4 + Term5 + Term6 + Term7 + Term8 + Term9
            
            # Exact Trace Expectation
            trace_term = torch.sum(Q_inv * M_j.transpose(1, 2), dim=(1, 2))
            ll_lat += torch.sum(-0.5 * log_det_Q - 0.5 * trace_term)
            
        # Priors
        active_Z = self.Z[self.tril_mask == 1]
        log_prior_Z = -0.5 * torch.sum(active_Z ** 2)
        log_prior_lin = -0.5 * (torch.sum(self.B**2) + torch.sum(self.C_int**2) + torch.sum(self.d_bias**2))
        log_prior_psi = -0.5 * torch.sum(self.log_psi ** 2)
        
        return ll_obs + ll_lat + log_prior_Z + log_prior_lin + log_prior_psi

    def pca_warm_start(self, subjects_data):
        with torch.no_grad():
            x_all = torch.cat([s['x'] for s in subjects_data], dim=0)
            x_valid = x_all[~torch.isnan(x_all).any(dim=1)] 
            U, S_vals, Vh = torch.linalg.svd(x_valid - x_valid.mean(dim=0), full_matrices=False)
            
            Lambda_pca = Vh[:self.K, :].T * torch.sqrt(S_vals[:self.K] / x_valid.shape[0])
            q, r = torch.linalg.qr(Lambda_pca.T)
            Lambda_tril = r.T * torch.sign(torch.diag(r.T)).unsqueeze(0)
            
            mask = self.tril_mask == 1
            self.Z.data[mask] = torch.log(torch.abs(Lambda_tril[mask]) + 1e-4)
            
            self.B.data.fill_(0.0)
            self.C_int.data.fill_(0.0)
            self.d_bias.data.fill_(0.0)
            self.log_psi.data.fill_(0.0)

    def fit_em_multistart(self, subjects_data, num_em_epochs=40, m_step_iters=20, lr=0.01, n_starts=5, burn_in_epochs=10):
        best_loss = float('inf')
        best_state_dict = None
        print(f"  Running {n_starts} Multi-Start Burn-ins ({burn_in_epochs} epochs each)...")
        
        for start in range(n_starts):
            with torch.no_grad():
                if self.theta_mode == "exact":
                    nn.init.normal_(self.L_S, mean=0.0, std=0.1)
                    self.L_S.data += torch.eye(self.K, device=self.Z.device)
                    nn.init.normal_(self.gamma_skew, mean=0.0, std=0.1)
                    nn.init.normal_(self.L_Omega, mean=0.0, std=0.1)
                    self.L_Omega.data += torch.eye(self.K, device=self.Z.device)
                else:
                    nn.init.normal_(self.log_rho, mean=-2.0, std=0.1)
                    
                nn.init.normal_(self.B, mean=0.0, std=0.1)
                nn.init.normal_(self.C_int, mean=0.0, std=0.1)
                nn.init.normal_(self.d_bias, mean=0.0, std=0.1)
            
            self.pca_warm_start(subjects_data)
            optimizer = optim.Adam(self.parameters(), lr=lr)
            start_loss = 0.0
            
            for epoch in range(burn_in_epochs):
                Theta, Omega = self.get_dynamics()
                Lambda = self.tril_mask * torch.exp(self.Z)
                
                smoothed_stats = []
                with torch.no_grad():
                    for subj in subjects_data:
                        A_trans, b_shift, dt, _, Q = self.get_subject_matrices(Theta, Omega, subj['u'], subj['t'])
                        smoothed_stats.append(self.kalman_smoother(subj['x'], A_trans, b_shift, dt, Lambda, Q))
                
                epoch_loss = 0.0
                for m in range(m_step_iters):
                    optimizer.zero_grad()
                    Theta_m, Omega_m = self.get_dynamics()
                    Lambda_m = self.tril_mask * torch.exp(self.Z)
                    loss = -self.expected_complete_log_posterior_vectorized(subjects_data, smoothed_stats, Theta_m, Omega_m, Lambda_m)
                    loss.backward()
                    optimizer.step()
                    epoch_loss += loss.item()
                start_loss = epoch_loss / m_step_iters
                
            if start_loss < best_loss:
                best_loss = start_loss
                best_state_dict = {k: v.clone() for k, v in self.state_dict().items()}
                
        print(f"  Burn-in complete. Training best initialization to convergence...")
        self.load_state_dict(best_state_dict)
        optimizer = optim.Adam(self.parameters(), lr=lr)
        
        for epoch in range(num_em_epochs - burn_in_epochs):
            Theta, Omega = self.get_dynamics()
            Lambda = self.tril_mask * torch.exp(self.Z)
            
            smoothed_stats = []
            with torch.no_grad():
                for subj in subjects_data:
                    A_trans, b_shift, dt, _, Q = self.get_subject_matrices(Theta, Omega, subj['u'], subj['t'])
                    smoothed_stats.append(self.kalman_smoother(subj['x'], A_trans, b_shift, dt, Lambda, Q))
            
            for m in range(m_step_iters):
                optimizer.zero_grad()
                Theta_m, Omega_m = self.get_dynamics()
                Lambda_m = self.tril_mask * torch.exp(self.Z)
                loss = -self.expected_complete_log_posterior_vectorized(subjects_data, smoothed_stats, Theta_m, Omega_m, Lambda_m)
                loss.backward()
                optimizer.step()
                
        return smoothed_stats

# ---------------------------------------------------------
# 3. Exact Data Simulation (Using Authentic Q)
# ---------------------------------------------------------
def simulate_ad_cohort_stress(N, D, K, C_dim, theta_mode="exact", seed=42):
    torch.manual_seed(seed)
    
    if theta_mode == "diagonal":
        rho_true = torch.linspace(0.02, 0.15, K)
        Theta_true = torch.diag(rho_true)
        Omega_true = torch.eye(K)
    else:
        # Generate Ground Truth using Exact Stan Mapping
        L_S_true = torch.tril(torch.randn(K, K) * 0.3 + torch.eye(K)*0.5)
        S_true = L_S_true @ L_S_true.T + 1e-4 * torch.eye(K)
        
        gamma_true = torch.randn(K, K) * 0.2
        A_true = gamma_true - gamma_true.T
        
        L_Omega_true = torch.tril(torch.randn(K, K) * 0.3 + torch.eye(K))
        Omega_true = L_Omega_true @ L_Omega_true.T + 1e-4 * torch.eye(K)
        
        Theta_true = (S_true + A_true) @ torch.linalg.inv(Omega_true)
        
    B_true, C_true, d_true = torch.randn(K, C_dim)*0.5, torch.randn(K, C_dim)*0.5, torch.randn(K)*0.5
    Z_true = torch.randn(D, K) - 1.0 
    Lambda_true = torch.tril(torch.ones(D, K)) * torch.exp(Z_true)
    
    subjects_data = []
    for _ in range(N):
        J_i = torch.randint(3, 6, (1,)).item()
        age_baseline = torch.rand(1) * 20 + 55
        dt = torch.rand(J_i - 1) * 3.5 + 1.5
        times = torch.cat([age_baseline, age_baseline + torch.cumsum(dt, dim=0)])
        
        t_scaled = (times - 70.0) / 10.0 
        u = torch.randn(J_i, C_dim)
        
        F_true = torch.zeros(J_i, K)
        F_true[0] = torch.randn(K) * 0.1
        
        for j in range(1, J_i):
            delta_t = times[j] - times[j-1]
            A_ij = torch.linalg.matrix_exp(-Theta_true * delta_t)
            mu_j = B_true @ u[j] + C_true @ (u[j] * t_scaled[j]) + d_true
            
            # Exact Noise Simulation
            if theta_mode == "diagonal":
                theta_diag = torch.diag(Theta_true)
                Q_diag = (1.0 - torch.exp(-2.0 * theta_diag * delta_t)) / (2.0 * theta_diag)
                Q_true = torch.diag(Q_diag)
            else:
                Q_true = Omega_true - A_ij @ Omega_true @ A_ij.T
                Q_true = 0.5 * (Q_true + Q_true.T) + 1e-5 * torch.eye(K)
                
            L_Q = torch.linalg.cholesky(Q_true)
            noise = L_Q @ torch.randn(K)
            
            F_true[j] = A_ij @ F_true[j-1] + ((torch.eye(K) - A_ij) @ mu_j) + noise
            
        X_obs = F_true @ Lambda_true.T + torch.randn(J_i, D) # Homoscedastic synthetic noise
        subjects_data.append({'x': X_obs, 'u': u, 't': t_scaled, 'F_true': F_true})
        
    return subjects_data, {'Lambda': Lambda_true, 'Theta': Theta_true}

# ---------------------------------------------------------
# 4. Execution Wrapper
# ---------------------------------------------------------
def run_stress_test(n_runs=2):
    scenarios = [
        {"name": "1. Sparse Exact",        "N": 50,  "D": 20,   "K": 3, "C": 2},
        {"name": "2. High-Dim Proteomics", "N": 100, "D": 200,  "K": 4, "C": 2},
    ]
    
    print(f"{'Scenario':<25} | {'Mode':<10} | {'Λ Corr (μ ± σ)':<15} | {'F Corr (μ ± σ)':<15} | {'Θ Corr (μ ± σ)':<15} | {'Avg Time (s)'}")
    print("-" * 105)
    
    modes = ["exact", "diagonal"]
    
    for s in scenarios:
        for mode in modes:
            l_corrs, f_corrs, theta_corrs, run_times = [], [], [], []
            
            for run_idx in range(n_runs):
                subjects_data, true_params = simulate_ad_cohort_stress(
                    s["N"], s["D"], s["K"], s["C"], theta_mode=mode, seed=200+run_idx
                )
                
                start_time = time.time()
                
                model = CLOUDS(obs_dim=s["D"], latent_dim=s["K"], covar_dim=s["C"], theta_mode=mode)
                model.pca_warm_start(subjects_data)
                smoothed_stats = model.fit_em_multistart(subjects_data, num_em_epochs=30, m_step_iters=15, lr=0.01)
                
                with torch.no_grad():
                    mask = model.tril_mask == 1
                    Lambda_est = model.tril_mask * torch.exp(model.Z)
                    Theta_est, _ = model.get_dynamics()
                    
                    f_true_flat = torch.cat([subj['F_true'] for subj in subjects_data], dim=0).numpy().flatten()
                    f_est_flat = torch.cat([stat[0] for stat in smoothed_stats], dim=0).numpy().flatten()
                    
                    f_corr = np.corrcoef(f_true_flat, f_est_flat)[0, 1]
                    l_corr = np.corrcoef(true_params['Lambda'][mask].numpy(), Lambda_est[mask].numpy())[0, 1]
                    
                    if mode == "diagonal":
                        th_true = torch.diag(true_params['Theta']).numpy()
                        th_est = torch.diag(Theta_est).cpu().numpy()
                    else:
                        th_true = true_params['Theta'].numpy().flatten()
                        th_est = Theta_est.cpu().numpy().flatten()
                    theta_corr = np.corrcoef(th_true, th_est)[0, 1]
                
                elapsed = time.time() - start_time
                l_corrs.append(l_corr)
                f_corrs.append(f_corr)
                theta_corrs.append(theta_corr)
                run_times.append(elapsed)
            
            l_str = f"{np.mean(l_corrs):.3f} ± {np.std(l_corrs):.3f}"
            f_str = f"{np.mean(f_corrs):.3f} ± {np.std(f_corrs):.3f}"
            th_str = f"{np.mean(theta_corrs):.3f} ± {np.std(theta_corrs):.3f}"
            print(f"{s['name']:<25} | {mode.capitalize():<10} | {l_str:<15} | {f_str:<15} | {th_str:<15} | {np.mean(run_times):>8.1f}")
        print("-" * 105)

if __name__ == "__main__":
    run_stress_test(n_runs=2)

Scenario                  | Mode       | Λ Corr (μ ± σ)  | F Corr (μ ± σ)  | Θ Corr (μ ± σ)  | Avg Time (s)
---------------------------------------------------------------------------------------------------------
  Running 5 Multi-Start Burn-ins (10 epochs each)...
  Burn-in complete. Training best initialization to convergence...
  Running 5 Multi-Start Burn-ins (10 epochs each)...
  Burn-in complete. Training best initialization to convergence...
1. Sparse Exact           | Exact      | 0.587 ± 0.390   | 0.372 ± 0.585   | 0.441 ± 0.446   |    170.3
  Running 5 Multi-Start Burn-ins (10 epochs each)...
  Burn-in complete. Training best initialization to convergence...
  Running 5 Multi-Start Burn-ins (10 epochs each)...
  Burn-in complete. Training best initialization to convergence...
1. Sparse Exact           | Diagonal   | 0.945 ± 0.034   | 0.843 ± 0.021   | 0.834 ± 0.066   |    131.7
--------------------------------------------------------------------------------------------------

In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import time

# ---------------------------------------------------------
# 1. Helper Function: Newton-Schulz Matrix Inversion
# ---------------------------------------------------------
def batched_newton_schulz_inverse(A, num_iters=6):
    B, N, _ = A.shape
    I = torch.eye(N, device=A.device).unsqueeze(0).expand(B, N, N)
    frob_norm_sq = torch.sum(A * A, dim=(-2, -1), keepdim=True)
    X = A.transpose(-2, -1) / (frob_norm_sq + 1e-6)
    
    for _ in range(num_iters):
        AX = torch.bmm(A, X)
        X = torch.bmm(X, (2.0 * I) - AX)
    return X

# ---------------------------------------------------------
# 2. CLOUDS Model (New Notation: Gamma, Phi, alpha)
# ---------------------------------------------------------
class CLOUDS(nn.Module):
    def __init__(self, obs_dim, latent_dim, covar_dim, delta=1e-4):
        super().__init__()
        self.D = obs_dim
        self.K = latent_dim
        self.C_dim = covar_dim
        self.delta = delta
        
        # 1. Continuous-Time Dynamics (Lyapunov Parameterization)
        # L_G: Lower triangular diffusion matrix G
        self.L_G = nn.Parameter(torch.tril(torch.eye(self.K) + 0.1 * torch.randn(self.K, self.K)))
        # gamma_skew: For skew-symmetric rotational component A
        self.gamma_skew = nn.Parameter(torch.randn(self.K, self.K) * 0.1)
        # L_Omega_unc: Unconstrained matrix to build the Correlation Matrix Omega
        self.L_Omega_unc = nn.Parameter(torch.tril(torch.eye(self.K) + 0.1 * torch.randn(self.K, self.K)))
            
        # 2. Target Mean Function: mu = (Phi*u + alpha) * t
        self.Phi_int = nn.Parameter(torch.randn(self.K, self.C_dim) * 0.1)
        self.alpha_bias = nn.Parameter(torch.randn(self.K) * 0.1)
        
        # 3. Factor Loadings (Structural PCA identifiable)
        self.Z = nn.Parameter(torch.randn(self.D, self.K) - 0.5) 
        self.register_buffer('tril_mask', torch.tril(torch.ones(self.D, self.K)))
        
        # 4. Heteroscedastic Observation Noise
        self.log_psi = nn.Parameter(torch.zeros(self.D)) 

    def get_dynamics(self):
        device = self.Z.device
        
        # --- A. Build Correlation Matrix (Omega) ---
        L_unc_tril = torch.tril(self.L_Omega_unc)
        # Row-normalize to force diagonals of Omega to exactly 1
        row_norms = torch.norm(L_unc_tril, dim=1, keepdim=True).clamp(min=1e-6)
        L_corr = L_unc_tril / row_norms
        Omega = L_corr @ L_corr.T
        
        # --- B. Build Diffusion (G) and Symmetric Component (S) ---
        G = torch.tril(self.L_G)
        S = 0.5 * (G @ G.T) + self.delta * torch.eye(self.K, device=device)
        
        # --- C. Build Skew-Symmetric (A) ---
        A_skew = self.gamma_skew - self.gamma_skew.T
        
        # --- D. Enforce Lyapunov: Gamma = (S + A) * Omega^-1 ---
        Omega_inv = batched_newton_schulz_inverse(Omega.unsqueeze(0), num_iters=6).squeeze(0)
        Gamma = (S + A_skew) @ Omega_inv
        
        return Gamma, Omega, G

    def get_subject_matrices(self, Gamma, Omega, u, times):
        dt = times[1:] - times[:-1]
        device = self.Z.device
        
        Gamma_batch = Gamma.unsqueeze(0).expand(dt.shape[0], self.K, self.K)
        A_trans = torch.linalg.matrix_exp(-Gamma_batch * dt.view(-1, 1, 1))
        
        u_t, t_val = u[1:], times[1:].unsqueeze(1)
        
        # EXACT Mean Function: mu = (Phi*u + alpha) * t
        mu = (u_t @ self.Phi_int.T + self.alpha_bias) * t_val
        
        I_batch = torch.eye(self.K, device=device).unsqueeze(0).expand(dt.shape[0], self.K, self.K)
        b_shift = torch.bmm(I_batch - A_trans, mu.unsqueeze(-1)).squeeze(-1)
        
        # EXACT Continuous Transition Covariance (Q)
        Omega_batch = Omega.unsqueeze(0).expand(dt.shape[0], self.K, self.K)
        A_trans_T = A_trans.transpose(1, 2)
        # Q = Omega - exp(-Gamma*dt) * Omega * exp(-Gamma^T*dt)
        Q = Omega_batch - torch.bmm(A_trans, torch.bmm(Omega_batch, A_trans_T))
        Q = 0.5 * (Q + Q.transpose(1, 2)) # Force symmetry
            
        Lambda = self.tril_mask * torch.exp(self.Z)
        return A_trans, b_shift, dt, Lambda, Q

    def kalman_smoother(self, x_obs, A_trans, b_shift, dt, Lambda, Q):
        T = x_obs.shape[0]
        device = x_obs.device
        
        f_pred, P_pred = torch.zeros(T, self.K, device=device), torch.zeros(T, self.K, self.K, device=device)
        f_filt, P_filt = torch.zeros(T, self.K, device=device), torch.zeros(T, self.K, self.K, device=device)
        f_filt[0], P_filt[0] = torch.zeros(self.K, device=device), torch.eye(self.K, device=device)
        
        R_mat = torch.diag(torch.exp(self.log_psi))
        I_k = torch.eye(self.K, device=device)
        
        for j in range(1, T):
            idx = j - 1
            f_pred[j] = A_trans[idx] @ f_filt[j-1] + b_shift[idx]
            P_pred[j] = A_trans[idx] @ P_filt[j-1] @ A_trans[idx].T + Q[idx]
            
            if torch.isnan(x_obs[j]).all():
                f_filt[j], P_filt[j] = f_pred[j], P_pred[j]
            else:
                x_pred = Lambda @ f_pred[j]
                S_t = Lambda @ P_pred[j] @ Lambda.T + R_mat
                K_gain = P_pred[j] @ Lambda.T @ torch.linalg.inv(S_t)
                f_filt[j] = f_pred[j] + K_gain @ (x_obs[j] - x_pred)
                P_filt[j] = (I_k - K_gain @ Lambda) @ P_pred[j]
            
        f_smooth, P_smooth, P_cross = torch.zeros_like(f_filt), torch.zeros_like(P_filt), torch.zeros_like(P_filt)
        f_smooth[-1], P_smooth[-1] = f_filt[-1], P_filt[-1]
        
        for j in range(T-2, -1, -1):
            J_t = P_filt[j] @ A_trans[j].T @ torch.linalg.inv(P_pred[j+1])
            f_smooth[j] = f_filt[j] + J_t @ (f_smooth[j+1] - f_pred[j+1])
            P_smooth[j] = P_filt[j] + J_t @ (P_smooth[j+1] - P_pred[j+1]) @ J_t.T
            P_cross[j+1] = J_t @ P_smooth[j+1]
            
        return f_smooth, P_smooth, P_cross

    def expected_complete_log_posterior_vectorized(self, subjects_data, smoothed_stats, Gamma, Omega, Lambda):
        ll_obs, ll_lat = 0.0, 0.0
        
        inv_psi = torch.exp(-self.log_psi)
        L_Psi_L = Lambda.T @ torch.diag(inv_psi) @ Lambda 
        
        for i, subj in enumerate(subjects_data):
            x_obs, u, times = subj['x'], subj['u'], subj['t']
            f_s, P_s, P_c = smoothed_stats[i]
            
            A_trans, b_shift, _, _, Q_exact = self.get_subject_matrices(Gamma, Omega, u, times)
            
            valid_mask = ~torch.isnan(x_obs).any(dim=1)
            if valid_mask.any():
                x_v, f_v, P_v = x_obs[valid_mask], f_s[valid_mask], P_s[valid_mask]
                trace_E = torch.sum(P_v * L_Psi_L.unsqueeze(0), dim=(1,2)) + torch.sum(f_v * (f_v @ L_Psi_L), dim=1)
                term_obs = torch.sum((x_v**2) * inv_psi, dim=1) - 2 * torch.sum(x_v * (f_v @ Lambda.T) * inv_psi, dim=1) + trace_E
                ll_obs += torch.sum(-0.5 * term_obs - 0.5 * torch.sum(self.log_psi))
                
            Q_stable = Q_exact + 1e-5 * torch.eye(self.K, device=Q_exact.device).unsqueeze(0)
            Q_inv = torch.linalg.inv(Q_stable)
            log_det_Q = torch.linalg.slogdet(Q_stable)[1]
            
            f_j, f_jm1 = f_s[1:], f_s[:-1]
            P_j, P_jm1, P_cj = P_s[1:], P_s[:-1], P_c[1:]
            
            E_jj = P_j + torch.bmm(f_j.unsqueeze(-1), f_j.unsqueeze(1))
            E_jjm1 = P_cj + torch.bmm(f_j.unsqueeze(-1), f_jm1.unsqueeze(1))
            E_jm1jm1 = P_jm1 + torch.bmm(f_jm1.unsqueeze(-1), f_jm1.unsqueeze(1))
            
            A_T = A_trans.transpose(1, 2)
            M_j = (E_jj 
                   - torch.bmm(E_jjm1, A_T) 
                   - torch.bmm(A_trans, E_jjm1.transpose(1, 2)) 
                   + torch.bmm(A_trans, torch.bmm(E_jm1jm1, A_T)))
            
            b_uns, b_T = b_shift.unsqueeze(-1), b_shift.unsqueeze(1)
            f_j_uns, f_j_T = f_j.unsqueeze(-1), f_j.unsqueeze(1)
            f_jm1_uns, f_jm1_T = f_jm1.unsqueeze(-1), f_jm1.unsqueeze(1)
            
            M_j += (- torch.bmm(f_j_uns, b_T) 
                    - torch.bmm(b_uns, f_j_T) 
                    + torch.bmm(A_trans, torch.bmm(f_jm1_uns, b_T)) 
                    + torch.bmm(b_uns, torch.bmm(f_jm1_T, A_T)) 
                    + torch.bmm(b_uns, b_T))
            
            trace_term = torch.sum(Q_inv * M_j.transpose(1, 2), dim=(1, 2))
            ll_lat += torch.sum(-0.5 * log_det_Q - 0.5 * trace_term)
            
        active_Z = self.Z[self.tril_mask == 1]
        log_prior_Z = -0.5 * torch.sum(active_Z ** 2)
        log_prior_lin = -0.5 * (torch.sum(self.Phi_int**2) + torch.sum(self.alpha_bias**2))
        log_prior_psi = -0.5 * torch.sum(self.log_psi ** 2)
        
        return ll_obs + ll_lat + log_prior_Z + log_prior_lin + log_prior_psi

    def pca_warm_start(self, subjects_data):
        with torch.no_grad():
            x_all = torch.cat([s['x'] for s in subjects_data], dim=0)
            x_valid = x_all[~torch.isnan(x_all).any(dim=1)] 
            U, S_vals, Vh = torch.linalg.svd(x_valid - x_valid.mean(dim=0), full_matrices=False)
            
            Lambda_pca = Vh[:self.K, :].T * torch.sqrt(S_vals[:self.K] / x_valid.shape[0])
            q, r = torch.linalg.qr(Lambda_pca.T)
            Lambda_tril = r.T * torch.sign(torch.diag(r.T)).unsqueeze(0)
            
            mask = self.tril_mask == 1
            self.Z.data[mask] = torch.log(torch.abs(Lambda_tril[mask]) + 1e-4)
            self.Phi_int.data.fill_(0.0); self.alpha_bias.data.fill_(0.0); self.log_psi.data.fill_(0.0)

    def fit_em_multistart(self, subjects_data, num_em_epochs=40, m_step_iters=20, lr=0.01, n_starts=5, burn_in_epochs=10):
        best_loss = float('inf')
        best_state_dict = None
        
        for start in range(n_starts):
            with torch.no_grad():
                nn.init.normal_(self.L_G, mean=0.0, std=0.1)
                self.L_G.data += torch.eye(self.K, device=self.Z.device)
                nn.init.normal_(self.gamma_skew, mean=0.0, std=0.1)
                nn.init.normal_(self.L_Omega_unc, mean=0.0, std=0.1)
                self.L_Omega_unc.data += torch.eye(self.K, device=self.Z.device)
                nn.init.normal_(self.Phi_int, mean=0.0, std=0.1)
                nn.init.normal_(self.alpha_bias, mean=0.0, std=0.1)
            
            self.pca_warm_start(subjects_data)
            optimizer = optim.Adam(self.parameters(), lr=lr)
            start_loss = 0.0
            
            for epoch in range(burn_in_epochs):
                Gamma, Omega, _ = self.get_dynamics()
                Lambda = self.tril_mask * torch.exp(self.Z)
                
                smoothed_stats = []
                with torch.no_grad():
                    for subj in subjects_data:
                        A_trans, b_shift, dt, _, Q = self.get_subject_matrices(Gamma, Omega, subj['u'], subj['t'])
                        smoothed_stats.append(self.kalman_smoother(subj['x'], A_trans, b_shift, dt, Lambda, Q))
                
                epoch_loss = 0.0
                for m in range(m_step_iters):
                    optimizer.zero_grad()
                    Gamma_m, Omega_m, _ = self.get_dynamics()
                    Lambda_m = self.tril_mask * torch.exp(self.Z)
                    loss = -self.expected_complete_log_posterior_vectorized(subjects_data, smoothed_stats, Gamma_m, Omega_m, Lambda_m)
                    loss.backward()
                    optimizer.step()
                    epoch_loss += loss.item()
                start_loss = epoch_loss / m_step_iters
                
            if start_loss < best_loss:
                best_loss = start_loss
                best_state_dict = {k: v.clone() for k, v in self.state_dict().items()}
                
        self.load_state_dict(best_state_dict)
        optimizer = optim.Adam(self.parameters(), lr=lr)
        
        for epoch in range(num_em_epochs - burn_in_epochs):
            Gamma, Omega, _ = self.get_dynamics()
            Lambda = self.tril_mask * torch.exp(self.Z)
            
            smoothed_stats = []
            with torch.no_grad():
                for subj in subjects_data:
                    A_trans, b_shift, dt, _, Q = self.get_subject_matrices(Gamma, Omega, subj['u'], subj['t'])
                    smoothed_stats.append(self.kalman_smoother(subj['x'], A_trans, b_shift, dt, Lambda, Q))
            
            for m in range(m_step_iters):
                optimizer.zero_grad()
                Gamma_m, Omega_m, _ = self.get_dynamics()
                Lambda_m = self.tril_mask * torch.exp(self.Z)
                loss = -self.expected_complete_log_posterior_vectorized(subjects_data, smoothed_stats, Gamma_m, Omega_m, Lambda_m)
                loss.backward()
                optimizer.step()
                
        return smoothed_stats

# ---------------------------------------------------------
# 3. Exact Data Simulation (Using True Lyapunov Math & New Notation)
# ---------------------------------------------------------
def simulate_ad_cohort_stress(N, D, K, C_dim, seed=42):
    torch.manual_seed(seed)
    
    # 1. Simulate Correlation Matrix (Omega)
    L_unc_true = torch.tril(torch.randn(K, K) * 0.3 + torch.eye(K))
    L_corr_true = L_unc_true / torch.norm(L_unc_true, dim=1, keepdim=True)
    Omega_true = L_corr_true @ L_corr_true.T
    
    # 2. Simulate Diffusion (G) and S
    L_G_true = torch.tril(torch.randn(K, K) * 0.3 + torch.eye(K)*0.5)
    G_true = L_G_true
    S_true = 0.5 * (G_true @ G_true.T) + 1e-4 * torch.eye(K)
    
    # 3. Simulate Skew (A)
    gamma_skew_true = torch.randn(K, K) * 0.2
    A_true = gamma_skew_true - gamma_skew_true.T
    
    # 4. Drift (Gamma)
    Gamma_true = (S_true + A_true) @ torch.linalg.inv(Omega_true)
        
    Phi_true, alpha_true = torch.randn(K, C_dim)*0.5, torch.randn(K)*0.5
    Z_true = torch.randn(D, K) - 1.0 
    Lambda_true = torch.tril(torch.ones(D, K)) * torch.exp(Z_true)
    
    subjects_data = []
    for _ in range(N):
        J_i = torch.randint(3, 6, (1,)).item()
        age_baseline = torch.rand(1) * 20 + 55
        dt = torch.rand(J_i - 1) * 3.5 + 1.5
        times = torch.cat([age_baseline, age_baseline + torch.cumsum(dt, dim=0)])
        
        t_scaled = (times - 70.0) / 10.0 
        u = torch.randn(J_i, C_dim)
        
        F_true = torch.zeros(J_i, K)
        F_true[0] = torch.randn(K) * 0.1
        
        for j in range(1, J_i):
            delta_t = times[j] - times[j-1]
            A_ij = torch.linalg.matrix_exp(-Gamma_true * delta_t)
            
            # Exact Target Mean: mu = (Phi*u + alpha) * t
            mu_j = (Phi_true @ u[j] + alpha_true) * t_scaled[j]
            
            Q_true = Omega_true - A_ij @ Omega_true @ A_ij.T
            Q_true = 0.5 * (Q_true + Q_true.T) + 1e-5 * torch.eye(K)
                
            L_Q = torch.linalg.cholesky(Q_true)
            noise = L_Q @ torch.randn(K)
            
            F_true[j] = A_ij @ F_true[j-1] + ((torch.eye(K) - A_ij) @ mu_j) + noise
            
        X_obs = F_true @ Lambda_true.T + torch.randn(J_i, D)
        subjects_data.append({'x': X_obs, 'u': u, 't': t_scaled, 'F_true': F_true})
        
    return subjects_data, {'Lambda': Lambda_true, 'Gamma': Gamma_true}

# ---------------------------------------------------------
# 4. Execution Wrapper
# ---------------------------------------------------------
def run_stress_test(n_runs=2):
    scenarios = [
        {"name": "1. Sparse Exact",        "N": 50,  "D": 20,   "K": 3, "C": 2},
        {"name": "2. High-Dim Proteomics", "N": 100, "D": 200,  "K": 4, "C": 2},
    ]
    
    print(f"{'Scenario':<25} | {'Λ Corr (μ ± σ)':<15} | {'F Corr (μ ± σ)':<15} | {'Γ Corr (μ ± σ)':<15} | {'Avg Time (s)'}")
    print("-" * 90)
    
    for s in scenarios:
        l_corrs, f_corrs, gamma_corrs, run_times = [], [], [], []
        
        for run_idx in range(n_runs):
            subjects_data, true_params = simulate_ad_cohort_stress(
                s["N"], s["D"], s["K"], s["C"], seed=200+run_idx
            )
            
            start_time = time.time()
            
            model = CLOUDS(obs_dim=s["D"], latent_dim=s["K"], covar_dim=s["C"])
            model.pca_warm_start(subjects_data)
            smoothed_stats = model.fit_em_multistart(subjects_data, num_em_epochs=30, m_step_iters=15, lr=0.01)
            
            with torch.no_grad():
                mask = model.tril_mask == 1
                Lambda_est = model.tril_mask * torch.exp(model.Z)
                Gamma_est, _, _ = model.get_dynamics()
                
                f_true_flat = torch.cat([subj['F_true'] for subj in subjects_data], dim=0).numpy().flatten()
                f_est_flat = torch.cat([stat[0] for stat in smoothed_stats], dim=0).numpy().flatten()
                
                f_corr = np.corrcoef(f_true_flat, f_est_flat)[0, 1]
                l_corr = np.corrcoef(true_params['Lambda'][mask].numpy(), Lambda_est[mask].numpy())[0, 1]
                
                g_true = true_params['Gamma'].numpy().flatten()
                g_est = Gamma_est.cpu().numpy().flatten()
                gamma_corr = np.corrcoef(g_true, g_est)[0, 1]
            
            elapsed = time.time() - start_time
            l_corrs.append(l_corr)
            f_corrs.append(f_corr)
            gamma_corrs.append(gamma_corr)
            run_times.append(elapsed)
        
        l_str = f"{np.mean(l_corrs):.3f} ± {np.std(l_corrs):.3f}"
        f_str = f"{np.mean(f_corrs):.3f} ± {np.std(f_corrs):.3f}"
        g_str = f"{np.mean(gamma_corrs):.3f} ± {np.std(gamma_corrs):.3f}"
        print(f"{s['name']:<25} | {l_str:<15} | {f_str:<15} | {g_str:<15} | {np.mean(run_times):>8.1f}")
    print("-" * 90)

if __name__ == "__main__":
    run_stress_test(n_runs=2)

Scenario                  | Λ Corr (μ ± σ)  | F Corr (μ ± σ)  | Γ Corr (μ ± σ)  | Avg Time (s)
------------------------------------------------------------------------------------------
1. Sparse Exact           | 0.521 ± 0.399   | 0.557 ± 0.376   | 0.629 ± 0.118   |    132.8
2. High-Dim Proteomics    | 0.730 ± 0.254   | 0.827 ± 0.158   | 0.477 ± 0.051   |    410.4
------------------------------------------------------------------------------------------


In [3]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import time

# ---------------------------------------------------------
# 1. Helper Function: Newton-Schulz Matrix Inversion
# ---------------------------------------------------------
def batched_newton_schulz_inverse(A, num_iters=6):
    B, N, _ = A.shape
    I = torch.eye(N, device=A.device).unsqueeze(0).expand(B, N, N)
    frob_norm_sq = torch.sum(A * A, dim=(-2, -1), keepdim=True)
    X = A.transpose(-2, -1) / (frob_norm_sq + 1e-6)
    
    for _ in range(num_iters):
        AX = torch.bmm(A, X)
        X = torch.bmm(X, (2.0 * I) - AX)
    return X

# ---------------------------------------------------------
# 2. CLOUDS Model (Systematic Priors + Post-Hoc Identifiability)
# ---------------------------------------------------------
class CLOUDS(nn.Module):
    def __init__(self, obs_dim, latent_dim, covar_dim, delta=1e-4, theta_mode="exact"):
        super().__init__()
        self.D = obs_dim
        self.K = latent_dim
        self.C_dim = covar_dim
        self.delta = delta
        self.theta_mode = theta_mode
        
        # 1. Continuous-Time Dynamics (Lyapunov Parameterization)
        if self.theta_mode == "exact":
            self.L_G = nn.Parameter(torch.tril(torch.eye(self.K) + 0.1 * torch.randn(self.K, self.K)))
            self.gamma_skew = nn.Parameter(torch.randn(self.K, self.K) * 0.1)
            # FIX 2: Unconstrained Omega to prevent the Adam manifold trap
            self.L_Omega_unc = nn.Parameter(torch.tril(torch.eye(self.K) + 0.1 * torch.randn(self.K, self.K)))
        else:
            self.log_rho = nn.Parameter(torch.randn(self.K) * 0.1 - 2.0)
            self.log_omega = nn.Parameter(torch.randn(self.K) * 0.1)
            
        # 2. Target Mean Function: mu = (Phi*u + alpha) * t
        self.Phi_int = nn.Parameter(torch.randn(self.K, self.C_dim) * 0.1)
        self.alpha_bias = nn.Parameter(torch.randn(self.K) * 0.1)
        
        # 3. Factor Loadings (Structural PCA identifiable)
        self.Z = nn.Parameter(torch.randn(self.D, self.K) - 0.5) 
        self.register_buffer('tril_mask', torch.tril(torch.ones(self.D, self.K)))
        
        # 4. Heteroscedastic Observation Noise
        self.log_psi = nn.Parameter(torch.zeros(self.D)) 

    def get_dynamics(self):
        device = self.Z.device
        if self.theta_mode == "exact":
            # Omega is unconstrained during training (allows arbitrary scale)
            L_unc_tril = torch.tril(self.L_Omega_unc)
            Omega = L_unc_tril @ L_unc_tril.T + self.delta * torch.eye(self.K, device=device)
            
            G = torch.tril(self.L_G)
            S = 0.5 * (G @ G.T) + self.delta * torch.eye(self.K, device=device)
            A_skew = self.gamma_skew - self.gamma_skew.T
            
            Omega_inv = batched_newton_schulz_inverse(Omega.unsqueeze(0), num_iters=6).squeeze(0)
            Gamma = (S + A_skew) @ Omega_inv
            return Gamma, Omega, G
        else:
            Gamma = torch.diag(torch.exp(self.log_rho))
            Omega = torch.diag(torch.exp(self.log_omega))
            # In diagonal OU: G = sqrt(2 * Gamma * Omega)
            G = torch.sqrt(2.0 * Gamma @ Omega)
            return Gamma, Omega, G

    @torch.no_grad()
    def get_identifiable_parameters(self):
        """
        FIX 2: Post-Hoc Standardization.
        Converts the freely scaled Omega into a strict Correlation Matrix
        and pushes the excess scale back into Gamma, Lambda, Phi, and alpha.
        """
        Gamma_est, Omega_est, _ = self.get_dynamics()
        Lambda_est = self.tril_mask * torch.exp(self.Z)
        
        stds = torch.sqrt(torch.diag(Omega_est))
        D = torch.diag(stds)
        D_inv = torch.diag(1.0 / stds)
        
        Omega_corr = D_inv @ Omega_est @ D_inv
        Gamma_scaled = D_inv @ Gamma_est @ D
        Lambda_scaled = Lambda_est @ D
        Phi_scaled = D_inv @ self.Phi_int
        alpha_scaled = D_inv @ self.alpha_bias
        
        return {
            "Omega_corr": Omega_corr,
            "Gamma": Gamma_scaled,
            "Lambda": Lambda_scaled,
            "Phi": Phi_scaled,
            "alpha": alpha_scaled
        }

    def get_subject_matrices(self, Gamma, Omega, u, times):
        dt = times[1:] - times[:-1]
        device = self.Z.device
        
        Gamma_batch = Gamma.unsqueeze(0).expand(dt.shape[0], self.K, self.K)
        A_trans = torch.linalg.matrix_exp(-Gamma_batch * dt.view(-1, 1, 1))
        
        u_t, t_val = u[1:], times[1:].unsqueeze(1)
        mu = (u_t @ self.Phi_int.T + self.alpha_bias) * t_val
        
        I_batch = torch.eye(self.K, device=device).unsqueeze(0).expand(dt.shape[0], self.K, self.K)
        b_shift = torch.bmm(I_batch - A_trans, mu.unsqueeze(-1)).squeeze(-1)
        
        Omega_batch = Omega.unsqueeze(0).expand(dt.shape[0], self.K, self.K)
        A_trans_T = A_trans.transpose(1, 2)
        # Exact Q for both modes (since Diagonal Omega is handled as matrix)
        Q = Omega_batch - torch.bmm(A_trans, torch.bmm(Omega_batch, A_trans_T))
        Q = 0.5 * (Q + Q.transpose(1, 2)) 
            
        Lambda = self.tril_mask * torch.exp(self.Z)
        return A_trans, b_shift, dt, Lambda, Q

    def kalman_smoother(self, x_obs, A_trans, b_shift, dt, Lambda, Q):
        T = x_obs.shape[0]
        device = x_obs.device
        
        f_pred, P_pred = torch.zeros(T, self.K, device=device), torch.zeros(T, self.K, self.K, device=device)
        f_filt, P_filt = torch.zeros(T, self.K, device=device), torch.zeros(T, self.K, self.K, device=device)
        f_filt[0], P_filt[0] = torch.zeros(self.K, device=device), torch.eye(self.K, device=device)
        
        R_mat = torch.diag(torch.exp(self.log_psi))
        I_k = torch.eye(self.K, device=device)
        
        for j in range(1, T):
            idx = j - 1
            f_pred[j] = A_trans[idx] @ f_filt[j-1] + b_shift[idx]
            P_pred[j] = A_trans[idx] @ P_filt[j-1] @ A_trans[idx].T + Q[idx]
            
            if torch.isnan(x_obs[j]).all():
                f_filt[j], P_filt[j] = f_pred[j], P_pred[j]
            else:
                x_pred = Lambda @ f_pred[j]
                S_t = Lambda @ P_pred[j] @ Lambda.T + R_mat
                K_gain = P_pred[j] @ Lambda.T @ torch.linalg.inv(S_t)
                f_filt[j] = f_pred[j] + K_gain @ (x_obs[j] - x_pred)
                P_filt[j] = (I_k - K_gain @ Lambda) @ P_pred[j]
            
        f_smooth, P_smooth, P_cross = torch.zeros_like(f_filt), torch.zeros_like(P_filt), torch.zeros_like(P_filt)
        f_smooth[-1], P_smooth[-1] = f_filt[-1], P_filt[-1]
        
        for j in range(T-2, -1, -1):
            J_t = P_filt[j] @ A_trans[j].T @ torch.linalg.inv(P_pred[j+1])
            f_smooth[j] = f_filt[j] + J_t @ (f_smooth[j+1] - f_pred[j+1])
            P_smooth[j] = P_filt[j] + J_t @ (P_smooth[j+1] - P_pred[j+1]) @ J_t.T
            P_cross[j+1] = J_t @ P_smooth[j+1]
            
        return f_smooth, P_smooth, P_cross

    def expected_complete_log_posterior_vectorized(self, subjects_data, smoothed_stats, Gamma, Omega, Lambda):
        ll_obs, ll_lat = 0.0, 0.0
        
        inv_psi = torch.exp(-self.log_psi)
        L_Psi_L = Lambda.T @ torch.diag(inv_psi) @ Lambda 
        
        for i, subj in enumerate(subjects_data):
            x_obs, u, times = subj['x'], subj['u'], subj['t']
            f_s, P_s, P_c = smoothed_stats[i]
            
            A_trans, b_shift, _, _, Q_exact = self.get_subject_matrices(Gamma, Omega, u, times)
            
            valid_mask = ~torch.isnan(x_obs).any(dim=1)
            if valid_mask.any():
                x_v, f_v, P_v = x_obs[valid_mask], f_s[valid_mask], P_s[valid_mask]
                trace_E = torch.sum(P_v * L_Psi_L.unsqueeze(0), dim=(1,2)) + torch.sum(f_v * (f_v @ L_Psi_L), dim=1)
                term_obs = torch.sum((x_v**2) * inv_psi, dim=1) - 2 * torch.sum(x_v * (f_v @ Lambda.T) * inv_psi, dim=1) + trace_E
                ll_obs += torch.sum(-0.5 * term_obs - 0.5 * torch.sum(self.log_psi))
                
            Q_stable = Q_exact + 1e-5 * torch.eye(self.K, device=Q_exact.device).unsqueeze(0)
            Q_inv = torch.linalg.inv(Q_stable)
            log_det_Q = torch.linalg.slogdet(Q_stable)[1]
            
            f_j, f_jm1 = f_s[1:], f_s[:-1]
            P_j, P_jm1, P_cj = P_s[1:], P_s[:-1], P_c[1:]
            
            E_jj = P_j + torch.bmm(f_j.unsqueeze(-1), f_j.unsqueeze(1))
            E_jjm1 = P_cj + torch.bmm(f_j.unsqueeze(-1), f_jm1.unsqueeze(1))
            E_jm1jm1 = P_jm1 + torch.bmm(f_jm1.unsqueeze(-1), f_jm1.unsqueeze(1))
            
            A_T = A_trans.transpose(1, 2)
            M_j = (E_jj 
                   - torch.bmm(E_jjm1, A_T) 
                   - torch.bmm(A_trans, E_jjm1.transpose(1, 2)) 
                   + torch.bmm(A_trans, torch.bmm(E_jm1jm1, A_T)))
            
            b_uns, b_T = b_shift.unsqueeze(-1), b_shift.unsqueeze(1)
            f_j_uns, f_j_T = f_j.unsqueeze(-1), f_j.unsqueeze(1)
            f_jm1_uns, f_jm1_T = f_jm1.unsqueeze(-1), f_jm1.unsqueeze(1)
            
            M_j += (- torch.bmm(f_j_uns, b_T) 
                    - torch.bmm(b_uns, f_j_T) 
                    + torch.bmm(A_trans, torch.bmm(f_jm1_uns, b_T)) 
                    + torch.bmm(b_uns, torch.bmm(f_jm1_T, A_T)) 
                    + torch.bmm(b_uns, b_T))
            
            trace_term = torch.sum(Q_inv * M_j.transpose(1, 2), dim=(1, 2))
            ll_lat += torch.sum(-0.5 * log_det_Q - 0.5 * trace_term)
            
        # FIX 1: Systematic Structural Priors
        log_prior_dyn = 0.0
        if self.theta_mode == "exact":
            # 1. LKJ Log-Determinant Prior (Forces Omega toward Identity)
            stds = torch.sqrt(torch.diag(Omega))
            D_inv = torch.diag(1.0 / stds)
            Omega_corr = D_inv @ Omega @ D_inv
            eta = 2.0 
            log_prior_dyn += (eta - 1.0) * torch.linalg.slogdet(Omega_corr + 1e-5*torch.eye(self.K, device=Omega.device))[1]
            
            # 2. Laplace (L1) Prior on Skew (Forces A to Zero)
            lambda_laplace = 5.0
            log_prior_dyn -= lambda_laplace * torch.sum(torch.abs(self.gamma_skew))
            
            # 3. Laplace (L1) Prior on Diffusion Off-Diagonals (Forces G to Diagonal)
            off_diag_G = torch.tril(self.L_G, diagonal=-1)
            log_prior_dyn -= lambda_laplace * torch.sum(torch.abs(off_diag_G))
        else:
            log_prior_dyn -= 0.5 * torch.sum(self.log_rho**2) + 0.5 * torch.sum(self.log_omega**2)

        active_Z = self.Z[self.tril_mask == 1]
        log_prior_Z = -0.5 * torch.sum(active_Z ** 2)
        log_prior_lin = -0.5 * (torch.sum(self.Phi_int**2) + torch.sum(self.alpha_bias**2))
        log_prior_psi = -0.5 * torch.sum(self.log_psi ** 2)
        
        return ll_obs + ll_lat + log_prior_dyn + log_prior_Z + log_prior_lin + log_prior_psi

    def pca_warm_start(self, subjects_data):
        with torch.no_grad():
            x_all = torch.cat([s['x'] for s in subjects_data], dim=0)
            x_valid = x_all[~torch.isnan(x_all).any(dim=1)] 
            U, S_vals, Vh = torch.linalg.svd(x_valid - x_valid.mean(dim=0), full_matrices=False)
            
            Lambda_pca = Vh[:self.K, :].T * torch.sqrt(S_vals[:self.K] / x_valid.shape[0])
            q, r = torch.linalg.qr(Lambda_pca.T)
            Lambda_tril = r.T * torch.sign(torch.diag(r.T)).unsqueeze(0)
            
            mask = self.tril_mask == 1
            self.Z.data[mask] = torch.log(torch.abs(Lambda_tril[mask]) + 1e-4)
            self.Phi_int.data.fill_(0.0); self.alpha_bias.data.fill_(0.0); self.log_psi.data.fill_(0.0)

    def fit_em_multistart(self, subjects_data, num_em_epochs=40, m_step_iters=20, lr=0.01, n_starts=5, burn_in_epochs=10):
        best_loss = float('inf')
        best_state_dict = None
        
        for start in range(n_starts):
            with torch.no_grad():
                if self.theta_mode == "exact":
                    nn.init.normal_(self.L_G, mean=0.0, std=0.1)
                    self.L_G.data += torch.eye(self.K, device=self.Z.device)
                    nn.init.normal_(self.gamma_skew, mean=0.0, std=0.1)
                    nn.init.normal_(self.L_Omega_unc, mean=0.0, std=0.1)
                    self.L_Omega_unc.data += torch.eye(self.K, device=self.Z.device)
                else:
                    nn.init.normal_(self.log_rho, mean=-2.0, std=0.1)
                    nn.init.normal_(self.log_omega, mean=0.0, std=0.1)
                
                nn.init.normal_(self.Phi_int, mean=0.0, std=0.1)
                nn.init.normal_(self.alpha_bias, mean=0.0, std=0.1)
            
            self.pca_warm_start(subjects_data)
            optimizer = optim.Adam(self.parameters(), lr=lr)
            start_loss = 0.0
            
            for epoch in range(burn_in_epochs):
                Gamma, Omega, _ = self.get_dynamics()
                Lambda = self.tril_mask * torch.exp(self.Z)
                
                smoothed_stats = []
                with torch.no_grad():
                    for subj in subjects_data:
                        A_trans, b_shift, dt, _, Q = self.get_subject_matrices(Gamma, Omega, subj['u'], subj['t'])
                        smoothed_stats.append(self.kalman_smoother(subj['x'], A_trans, b_shift, dt, Lambda, Q))
                
                epoch_loss = 0.0
                for m in range(m_step_iters):
                    optimizer.zero_grad()
                    Gamma_m, Omega_m, _ = self.get_dynamics()
                    Lambda_m = self.tril_mask * torch.exp(self.Z)
                    loss = -self.expected_complete_log_posterior_vectorized(subjects_data, smoothed_stats, Gamma_m, Omega_m, Lambda_m)
                    loss.backward()
                    optimizer.step()
                    epoch_loss += loss.item()
                start_loss = epoch_loss / m_step_iters
                
            if start_loss < best_loss:
                best_loss = start_loss
                best_state_dict = {k: v.clone() for k, v in self.state_dict().items()}
                
        self.load_state_dict(best_state_dict)
        optimizer = optim.Adam(self.parameters(), lr=lr)
        
        for epoch in range(num_em_epochs - burn_in_epochs):
            Gamma, Omega, _ = self.get_dynamics()
            Lambda = self.tril_mask * torch.exp(self.Z)
            
            smoothed_stats = []
            with torch.no_grad():
                for subj in subjects_data:
                    A_trans, b_shift, dt, _, Q = self.get_subject_matrices(Gamma, Omega, subj['u'], subj['t'])
                    smoothed_stats.append(self.kalman_smoother(subj['x'], A_trans, b_shift, dt, Lambda, Q))
            
            for m in range(m_step_iters):
                optimizer.zero_grad()
                Gamma_m, Omega_m, _ = self.get_dynamics()
                Lambda_m = self.tril_mask * torch.exp(self.Z)
                loss = -self.expected_complete_log_posterior_vectorized(subjects_data, smoothed_stats, Gamma_m, Omega_m, Lambda_m)
                loss.backward()
                optimizer.step()
                
        return smoothed_stats

# ---------------------------------------------------------
# 3. Authentic Data Simulation Wrapper
# ---------------------------------------------------------
def simulate_ad_cohort_stress(N, D, K, C_dim, theta_mode="exact", seed=42):
    torch.manual_seed(seed)
    
    if theta_mode == "diagonal":
        rho_true = torch.linspace(0.02, 0.15, K)
        omega_true = torch.ones(K)
        Gamma_true = torch.diag(rho_true)
        Omega_true = torch.diag(omega_true)
    else:
        # Simulate exact data such that Omega is ALREADY a strict correlation matrix
        L_unc_true = torch.tril(torch.randn(K, K) * 0.3 + torch.eye(K))
        L_corr_true = L_unc_true / torch.norm(L_unc_true, dim=1, keepdim=True)
        Omega_true = L_corr_true @ L_corr_true.T
        
        L_G_true = torch.tril(torch.randn(K, K) * 0.3 + torch.eye(K)*0.5)
        S_true = 0.5 * (L_G_true @ L_G_true.T) + 1e-4 * torch.eye(K)
        
        gamma_skew_true = torch.randn(K, K) * 0.2
        A_true = gamma_skew_true - gamma_skew_true.T
        Gamma_true = (S_true + A_true) @ torch.linalg.inv(Omega_true)
        
    Phi_true, alpha_true = torch.randn(K, C_dim)*0.5, torch.randn(K)*0.5
    Z_true = torch.randn(D, K) - 1.0 
    Lambda_true = torch.tril(torch.ones(D, K)) * torch.exp(Z_true)
    
    subjects_data = []
    for _ in range(N):
        J_i = torch.randint(3, 6, (1,)).item()
        age_baseline = torch.rand(1) * 20 + 55
        dt = torch.rand(J_i - 1) * 3.5 + 1.5
        times = torch.cat([age_baseline, age_baseline + torch.cumsum(dt, dim=0)])
        
        t_scaled = (times - 70.0) / 10.0 
        u = torch.randn(J_i, C_dim)
        
        F_true = torch.zeros(J_i, K)
        F_true[0] = torch.randn(K) * 0.1
        
        for j in range(1, J_i):
            delta_t = times[j] - times[j-1]
            A_ij = torch.linalg.matrix_exp(-Gamma_true * delta_t)
            mu_j = (Phi_true @ u[j] + alpha_true) * t_scaled[j]
            
            Q_true = Omega_true - A_ij @ Omega_true @ A_ij.T
            Q_true = 0.5 * (Q_true + Q_true.T) + 1e-5 * torch.eye(K)
            L_Q = torch.linalg.cholesky(Q_true)
            noise = L_Q @ torch.randn(K)
            
            F_true[j] = A_ij @ F_true[j-1] + ((torch.eye(K) - A_ij) @ mu_j) + noise
            
        X_obs = F_true @ Lambda_true.T + torch.randn(J_i, D)
        subjects_data.append({'x': X_obs, 'u': u, 't': t_scaled, 'F_true': F_true})
        
    return subjects_data, {'Lambda': Lambda_true, 'Gamma': Gamma_true}

# ---------------------------------------------------------
# 4. Apples-to-Apples Evaluation Wrapper
# ---------------------------------------------------------
def run_stress_test(n_runs=2):
    scenarios = [
        {"name": "1. Sparse Test",         "N": 50,  "D": 20,   "K": 3, "C": 2},
        {"name": "2. High-Dim Proteomics", "N": 100, "D": 200,  "K": 4, "C": 2},
    ]
    
    print(f"{'Scenario':<25} | {'Mode':<10} | {'Λ Corr (μ ± σ)':<15} | {'F Corr (μ ± σ)':<15} | {'Γ Corr (μ ± σ)':<15} | {'Avg Time (s)'}")
    print("-" * 105)
    
    for s in scenarios:
        for mode in ["exact", "diagonal"]:
            l_corrs, f_corrs, gamma_corrs, run_times = [], [], [], []
            
            for run_idx in range(n_runs):
                subjects_data, true_params = simulate_ad_cohort_stress(
                    s["N"], s["D"], s["K"], s["C"], theta_mode=mode, seed=300+run_idx
                )
                
                start_time = time.time()
                
                model = CLOUDS(obs_dim=s["D"], latent_dim=s["K"], covar_dim=s["C"], theta_mode=mode)
                model.pca_warm_start(subjects_data)
                smoothed_stats = model.fit_em_multistart(subjects_data, num_em_epochs=30, m_step_iters=15, lr=0.01)
                
                with torch.no_grad():
                    # Evaluate on Identifiable parameters!
                    identifiable = model.get_identifiable_parameters()
                    Lambda_est = identifiable["Lambda"]
                    Gamma_est = identifiable["Gamma"]
                    mask = model.tril_mask == 1
                    
                    f_true_flat = torch.cat([subj['F_true'] for subj in subjects_data], dim=0).numpy().flatten()
                    f_est_flat = torch.cat([stat[0] for stat in smoothed_stats], dim=0).numpy().flatten()
                    
                    f_corr = np.corrcoef(f_true_flat, f_est_flat)[0, 1]
                    l_corr = np.corrcoef(true_params['Lambda'][mask].numpy(), Lambda_est[mask].numpy())[0, 1]
                    
                    if mode == "diagonal":
                        g_true = torch.diag(true_params['Gamma']).numpy()
                        g_est = torch.diag(Gamma_est).cpu().numpy()
                    else:
                        g_true = true_params['Gamma'].numpy().flatten()
                        g_est = Gamma_est.cpu().numpy().flatten()
                        
                    gamma_corr = np.corrcoef(g_true, g_est)[0, 1]
                
                elapsed = time.time() - start_time
                l_corrs.append(l_corr)
                f_corrs.append(f_corr)
                gamma_corrs.append(gamma_corr)
                run_times.append(elapsed)
            
            l_str = f"{np.mean(l_corrs):.3f} ± {np.std(l_corrs):.3f}"
            f_str = f"{np.mean(f_corrs):.3f} ± {np.std(f_corrs):.3f}"
            g_str = f"{np.mean(gamma_corrs):.3f} ± {np.std(gamma_corrs):.3f}"
            print(f"{s['name']:<25} | {mode.capitalize():<10} | {l_str:<15} | {f_str:<15} | {g_str:<15} | {np.mean(run_times):>8.1f}")
        print("-" * 105)

if __name__ == "__main__":
    run_stress_test(n_runs=2)

Scenario                  | Mode       | Λ Corr (μ ± σ)  | F Corr (μ ± σ)  | Γ Corr (μ ± σ)  | Avg Time (s)
---------------------------------------------------------------------------------------------------------
1. Sparse Test            | Exact      | 0.972 ± 0.017   | 0.910 ± 0.015   | 0.621 ± 0.233   |    134.7
1. Sparse Test            | Diagonal   | 0.944 ± 0.028   | 0.901 ± 0.025   | 0.930 ± 0.051   |    126.9
---------------------------------------------------------------------------------------------------------
2. High-Dim Proteomics    | Exact      | 0.497 ± 0.234   | 0.398 ± 0.177   | 0.582 ± 0.196   |    404.7
2. High-Dim Proteomics    | Diagonal   | 0.761 ± 0.179   | 0.641 ± 0.304   | 0.955 ± 0.042   |    396.8
---------------------------------------------------------------------------------------------------------


In [6]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import time

# ---------------------------------------------------------
# 1. Helper Function: Newton-Schulz Matrix Inversion
# ---------------------------------------------------------
def batched_newton_schulz_inverse(A, num_iters=6):
    B, N, _ = A.shape
    I = torch.eye(N, device=A.device).unsqueeze(0).expand(B, N, N)
    frob_norm_sq = torch.sum(A * A, dim=(-2, -1), keepdim=True)
    X = A.transpose(-2, -1) / (frob_norm_sq + 1e-6)
    
    for _ in range(num_iters):
        AX = torch.bmm(A, X)
        X = torch.bmm(X, (2.0 * I) - AX)
    return X

# ---------------------------------------------------------
# 2. CLOUDS Model (Scaled Priors + Staged Warmup)
# ---------------------------------------------------------
class CLOUDS(nn.Module):
    def __init__(self, obs_dim, latent_dim, covar_dim, delta=1e-4, theta_mode="exact"):
        super().__init__()
        self.D = obs_dim
        self.K = latent_dim
        self.C_dim = covar_dim
        self.delta = delta
        self.theta_mode = theta_mode
        
        # 1. Continuous-Time Dynamics (Lyapunov Parameterization)
        if self.theta_mode == "exact":
            self.L_G = nn.Parameter(torch.tril(torch.eye(self.K) + 0.1 * torch.randn(self.K, self.K)))
            self.gamma_skew = nn.Parameter(torch.randn(self.K, self.K) * 0.1)
            self.L_Omega_unc = nn.Parameter(torch.tril(torch.eye(self.K) + 0.1 * torch.randn(self.K, self.K)))
        else:
            self.log_rho = nn.Parameter(torch.randn(self.K) * 0.1 - 2.0)
            self.log_omega = nn.Parameter(torch.randn(self.K) * 0.1)
            
        # 2. Target Mean Function: mu = (Phi*u + alpha) * t
        self.Phi_int = nn.Parameter(torch.randn(self.K, self.C_dim) * 0.1)
        self.alpha_bias = nn.Parameter(torch.randn(self.K) * 0.1)
        
        # 3. Factor Loadings (Structural PCA identifiable)
        self.Z = nn.Parameter(torch.randn(self.D, self.K) - 0.5) 
        self.register_buffer('tril_mask', torch.tril(torch.ones(self.D, self.K)))
        
        # 4. Heteroscedastic Observation Noise
        self.log_psi = nn.Parameter(torch.zeros(self.D)) 

    def get_dynamics(self):
        device = self.Z.device
        if self.theta_mode == "exact":
            L_unc_tril = torch.tril(self.L_Omega_unc)
            Omega = L_unc_tril @ L_unc_tril.T + self.delta * torch.eye(self.K, device=device)
            
            G = torch.tril(self.L_G)
            S = 0.5 * (G @ G.T) + self.delta * torch.eye(self.K, device=device)
            A_skew = self.gamma_skew - self.gamma_skew.T
            
            Omega_inv = batched_newton_schulz_inverse(Omega.unsqueeze(0), num_iters=6).squeeze(0)
            Gamma = (S + A_skew) @ Omega_inv
            return Gamma, Omega, G
        else:
            Gamma = torch.diag(torch.exp(self.log_rho))
            Omega = torch.diag(torch.exp(self.log_omega))
            G = torch.sqrt(2.0 * Gamma @ Omega)
            return Gamma, Omega, G

    @torch.no_grad()
    def get_identifiable_parameters(self):
        Gamma_est, Omega_est, _ = self.get_dynamics()
        Lambda_est = self.tril_mask * torch.exp(self.Z)
        
        stds = torch.sqrt(torch.diag(Omega_est))
        D = torch.diag(stds)
        D_inv = torch.diag(1.0 / stds)
        
        Omega_corr = D_inv @ Omega_est @ D_inv
        Gamma_scaled = D_inv @ Gamma_est @ D
        Lambda_scaled = Lambda_est @ D
        Phi_scaled = D_inv @ self.Phi_int
        alpha_scaled = D_inv @ self.alpha_bias
        
        return {
            "Omega_corr": Omega_corr,
            "Gamma": Gamma_scaled,
            "Lambda": Lambda_scaled,
            "Phi": Phi_scaled,
            "alpha": alpha_scaled
        }

    def get_subject_matrices(self, Gamma, Omega, u, times):
        dt = times[1:] - times[:-1]
        device = self.Z.device
        
        Gamma_batch = Gamma.unsqueeze(0).expand(dt.shape[0], self.K, self.K)
        A_trans = torch.linalg.matrix_exp(-Gamma_batch * dt.view(-1, 1, 1))
        
        u_t, t_val = u[1:], times[1:].unsqueeze(1)
        mu = (u_t @ self.Phi_int.T + self.alpha_bias) * t_val
        
        I_batch = torch.eye(self.K, device=device).unsqueeze(0).expand(dt.shape[0], self.K, self.K)
        b_shift = torch.bmm(I_batch - A_trans, mu.unsqueeze(-1)).squeeze(-1)
        
        Omega_batch = Omega.unsqueeze(0).expand(dt.shape[0], self.K, self.K)
        A_trans_T = A_trans.transpose(1, 2)
        Q = Omega_batch - torch.bmm(A_trans, torch.bmm(Omega_batch, A_trans_T))
        Q = 0.5 * (Q + Q.transpose(1, 2)) 
            
        Lambda = self.tril_mask * torch.exp(self.Z)
        return A_trans, b_shift, dt, Lambda, Q

    def kalman_smoother(self, x_obs, A_trans, b_shift, dt, Lambda, Q):
        T = x_obs.shape[0]
        device = x_obs.device
        
        f_pred, P_pred = torch.zeros(T, self.K, device=device), torch.zeros(T, self.K, self.K, device=device)
        f_filt, P_filt = torch.zeros(T, self.K, device=device), torch.zeros(T, self.K, self.K, device=device)
        f_filt[0], P_filt[0] = torch.zeros(self.K, device=device), torch.eye(self.K, device=device)
        
        R_mat = torch.diag(torch.exp(self.log_psi))
        I_k = torch.eye(self.K, device=device)
        
        for j in range(1, T):
            idx = j - 1
            f_pred[j] = A_trans[idx] @ f_filt[j-1] + b_shift[idx]
            P_pred[j] = A_trans[idx] @ P_filt[j-1] @ A_trans[idx].T + Q[idx]
            
            if torch.isnan(x_obs[j]).all():
                f_filt[j], P_filt[j] = f_pred[j], P_pred[j]
            else:
                x_pred = Lambda @ f_pred[j]
                S_t = Lambda @ P_pred[j] @ Lambda.T + R_mat
                K_gain = P_pred[j] @ Lambda.T @ torch.linalg.inv(S_t)
                f_filt[j] = f_pred[j] + K_gain @ (x_obs[j] - x_pred)
                P_filt[j] = (I_k - K_gain @ Lambda) @ P_pred[j]
            
        f_smooth, P_smooth, P_cross = torch.zeros_like(f_filt), torch.zeros_like(P_filt), torch.zeros_like(P_filt)
        f_smooth[-1], P_smooth[-1] = f_filt[-1], P_filt[-1]
        
        for j in range(T-2, -1, -1):
            J_t = P_filt[j] @ A_trans[j].T @ torch.linalg.inv(P_pred[j+1])
            f_smooth[j] = f_filt[j] + J_t @ (f_smooth[j+1] - f_pred[j+1])
            P_smooth[j] = P_filt[j] + J_t @ (P_smooth[j+1] - P_pred[j+1]) @ J_t.T
            P_cross[j+1] = J_t @ P_smooth[j+1]
            
        return f_smooth, P_smooth, P_cross

    def expected_complete_log_posterior_vectorized(self, subjects_data, smoothed_stats, Gamma, Omega, Lambda):
        ll_obs, ll_lat = 0.0, 0.0
        
        inv_psi = torch.exp(-self.log_psi)
        L_Psi_L = Lambda.T @ torch.diag(inv_psi) @ Lambda 
        
        # Calculate total observations to scale priors correctly
        total_obs = sum([subj['x'].shape[0] for subj in subjects_data])
        
        for i, subj in enumerate(subjects_data):
            x_obs, u, times = subj['x'], subj['u'], subj['t']
            f_s, P_s, P_c = smoothed_stats[i]
            
            A_trans, b_shift, _, _, Q_exact = self.get_subject_matrices(Gamma, Omega, u, times)
            
            valid_mask = ~torch.isnan(x_obs).any(dim=1)
            if valid_mask.any():
                x_v, f_v, P_v = x_obs[valid_mask], f_s[valid_mask], P_s[valid_mask]
                trace_E = torch.sum(P_v * L_Psi_L.unsqueeze(0), dim=(1,2)) + torch.sum(f_v * (f_v @ L_Psi_L), dim=1)
                term_obs = torch.sum((x_v**2) * inv_psi, dim=1) - 2 * torch.sum(x_v * (f_v @ Lambda.T) * inv_psi, dim=1) + trace_E
                ll_obs += torch.sum(-0.5 * term_obs - 0.5 * torch.sum(self.log_psi))
                
            Q_stable = Q_exact + 1e-5 * torch.eye(self.K, device=Q_exact.device).unsqueeze(0)
            Q_inv = torch.linalg.inv(Q_stable)
            log_det_Q = torch.linalg.slogdet(Q_stable)[1]
            
            f_j, f_jm1 = f_s[1:], f_s[:-1]
            P_j, P_jm1, P_cj = P_s[1:], P_s[:-1], P_c[1:]
            
            E_jj = P_j + torch.bmm(f_j.unsqueeze(-1), f_j.unsqueeze(1))
            E_jjm1 = P_cj + torch.bmm(f_j.unsqueeze(-1), f_jm1.unsqueeze(1))
            E_jm1jm1 = P_jm1 + torch.bmm(f_jm1.unsqueeze(-1), f_jm1.unsqueeze(1))
            
            A_T = A_trans.transpose(1, 2)
            M_j = (E_jj 
                   - torch.bmm(E_jjm1, A_T) 
                   - torch.bmm(A_trans, E_jjm1.transpose(1, 2)) 
                   + torch.bmm(A_trans, torch.bmm(E_jm1jm1, A_T)))
            
            b_uns, b_T = b_shift.unsqueeze(-1), b_shift.unsqueeze(1)
            f_j_uns, f_j_T = f_j.unsqueeze(-1), f_j.unsqueeze(1)
            f_jm1_uns, f_jm1_T = f_jm1.unsqueeze(-1), f_jm1.unsqueeze(1)
            
            M_j += (- torch.bmm(f_j_uns, b_T) 
                    - torch.bmm(b_uns, f_j_T) 
                    + torch.bmm(A_trans, torch.bmm(f_jm1_uns, b_T)) 
                    + torch.bmm(b_uns, torch.bmm(f_jm1_T, A_T)) 
                    + torch.bmm(b_uns, b_T))
            
            trace_term = torch.sum(Q_inv * M_j.transpose(1, 2), dim=(1, 2))
            ll_lat += torch.sum(-0.5 * log_det_Q - 0.5 * trace_term)
            
        # Scaled Systematic Structural Priors
        log_prior_dyn = 0.0
        if self.theta_mode == "exact":
            stds = torch.sqrt(torch.diag(Omega))
            D_inv = torch.diag(1.0 / stds)
            Omega_corr = D_inv @ Omega @ D_inv
            
            # 1. LKJ Prior: Keep it gentle (eta = 1.5 instead of 2.0)
            eta = 1.5 
            # REMOVED total_obs multiplier
            log_prior_dyn += (eta - 1.0) * torch.linalg.slogdet(Omega_corr + 1e-5*torch.eye(self.K, device=Omega.device))[1]
            
            # 2. Laplace Sparsity Prior: Dramatically reduce the penalty
            lambda_laplace = 0.1 # Reduced from 1.0, and REMOVED total_obs multiplier
            log_prior_dyn -= lambda_laplace * torch.sum(torch.abs(self.gamma_skew))
            
            off_diag_G = torch.tril(self.L_G, diagonal=-1)
            log_prior_dyn -= lambda_laplace * torch.sum(torch.abs(off_diag_G))
        else:
            log_prior_dyn -= 0.5 * torch.sum(self.log_rho**2) + 0.5 * torch.sum(self.log_omega**2)

        active_Z = self.Z[self.tril_mask == 1]
        log_prior_Z = -0.5 * torch.sum(active_Z ** 2)
        log_prior_lin = -0.5 * (torch.sum(self.Phi_int**2) + torch.sum(self.alpha_bias**2))
        log_prior_psi = -0.5 * torch.sum(self.log_psi ** 2)
        
        return ll_obs + ll_lat + log_prior_dyn + log_prior_Z + log_prior_lin + log_prior_psi

    def pca_warm_start(self, subjects_data):
        with torch.no_grad():
            x_all = torch.cat([s['x'] for s in subjects_data], dim=0)
            x_valid = x_all[~torch.isnan(x_all).any(dim=1)] 
            U, S_vals, Vh = torch.linalg.svd(x_valid - x_valid.mean(dim=0), full_matrices=False)
            
            Lambda_pca = Vh[:self.K, :].T * torch.sqrt(S_vals[:self.K] / x_valid.shape[0])
            q, r = torch.linalg.qr(Lambda_pca.T)
            Lambda_tril = r.T * torch.sign(torch.diag(r.T)).unsqueeze(0)
            
            mask = self.tril_mask == 1
            self.Z.data[mask] = torch.log(torch.abs(Lambda_tril[mask]) + 1e-4)
            self.Phi_int.data.fill_(0.0); self.alpha_bias.data.fill_(0.0); self.log_psi.data.fill_(0.0)

    def fit_em_multistart(self, subjects_data, num_em_epochs=40, warmup_epochs=15, m_step_iters=20, lr=0.01, n_starts=5, burn_in_epochs=10):
        best_loss = float('inf')
        best_state_dict = None
        
        for start in range(n_starts):
            with torch.no_grad():
                if self.theta_mode == "exact":
                    nn.init.normal_(self.L_G, mean=0.0, std=0.1)
                    self.L_G.data += torch.eye(self.K, device=self.Z.device)
                    nn.init.normal_(self.gamma_skew, mean=0.0, std=0.1)
                    nn.init.normal_(self.L_Omega_unc, mean=0.0, std=0.1)
                    self.L_Omega_unc.data += torch.eye(self.K, device=self.Z.device)
                else:
                    nn.init.normal_(self.log_rho, mean=-2.0, std=0.1)
                    nn.init.normal_(self.log_omega, mean=0.0, std=0.1)
                
                nn.init.normal_(self.Phi_int, mean=0.0, std=0.1)
                nn.init.normal_(self.alpha_bias, mean=0.0, std=0.1)
            
            self.pca_warm_start(subjects_data)
            optimizer = optim.Adam(self.parameters(), lr=lr)
            start_loss = 0.0
            
            for epoch in range(burn_in_epochs):
                Gamma, Omega, _ = self.get_dynamics()
                Lambda = self.tril_mask * torch.exp(self.Z)
                
                smoothed_stats = []
                with torch.no_grad():
                    for subj in subjects_data:
                        A_trans, b_shift, dt, _, Q = self.get_subject_matrices(Gamma, Omega, subj['u'], subj['t'])
                        smoothed_stats.append(self.kalman_smoother(subj['x'], A_trans, b_shift, dt, Lambda, Q))
                
                epoch_loss = 0.0
                for m in range(m_step_iters):
                    optimizer.zero_grad()
                    Gamma_m, Omega_m, _ = self.get_dynamics()
                    Lambda_m = self.tril_mask * torch.exp(self.Z)
                    loss = -self.expected_complete_log_posterior_vectorized(subjects_data, smoothed_stats, Gamma_m, Omega_m, Lambda_m)
                    loss.backward()
                    torch.nn.utils.clip_grad_norm_(self.parameters(), max_norm=5.0)
                    optimizer.step()
                    epoch_loss += loss.item()
                start_loss = epoch_loss / m_step_iters
                
            if start_loss < best_loss:
                best_loss = start_loss
                best_state_dict = {k: v.clone() for k, v in self.state_dict().items()}
                
        self.load_state_dict(best_state_dict)
        
        # Freeze BOTH Spatial Loadings (Z) and Spatial Noise (log_psi) during warmup
        temporal_params = [p for n, p in self.named_parameters() if n not in ['Z', 'log_psi']]
        opt_dynamics_only = optim.Adam(temporal_params, lr=lr)
        
        # Joint optimizer for after the warmup
        opt_joint = optim.Adam(self.parameters(), lr=lr)
        
        for epoch in range(num_em_epochs - burn_in_epochs):
            Gamma, Omega, _ = self.get_dynamics()
            Lambda = self.tril_mask * torch.exp(self.Z)
            
            smoothed_stats = []
            with torch.no_grad():
                for subj in subjects_data:
                    A_trans, b_shift, dt, _, Q = self.get_subject_matrices(Gamma, Omega, subj['u'], subj['t'])
                    smoothed_stats.append(self.kalman_smoother(subj['x'], A_trans, b_shift, dt, Lambda, Q))
            
            active_optimizer = opt_dynamics_only if epoch < warmup_epochs else opt_joint
            
            for m in range(m_step_iters):
                active_optimizer.zero_grad()
                Gamma_m, Omega_m, _ = self.get_dynamics()
                Lambda_m = self.tril_mask * torch.exp(self.Z)
                loss = -self.expected_complete_log_posterior_vectorized(subjects_data, smoothed_stats, Gamma_m, Omega_m, Lambda_m)
                loss.backward()
                torch.nn.utils.clip_grad_norm_(self.parameters(), max_norm=5.0)
                active_optimizer.step()
                
        return smoothed_stats

# ---------------------------------------------------------
# 3. Authentic Data Simulation Wrapper
# ---------------------------------------------------------
def simulate_ad_cohort_stress(N, D, K, C_dim, theta_mode="exact", seed=42):
    torch.manual_seed(seed)
    
    if theta_mode == "diagonal":
        rho_true = torch.linspace(0.02, 0.15, K)
        omega_true = torch.ones(K)
        Gamma_true = torch.diag(rho_true)
        Omega_true = torch.diag(omega_true)
    else:
        L_unc_true = torch.tril(torch.randn(K, K) * 0.3 + torch.eye(K))
        L_corr_true = L_unc_true / torch.norm(L_unc_true, dim=1, keepdim=True)
        Omega_true = L_corr_true @ L_corr_true.T
        
        L_G_true = torch.tril(torch.randn(K, K) * 0.3 + torch.eye(K)*0.5)
        S_true = 0.5 * (L_G_true @ L_G_true.T) + 1e-4 * torch.eye(K)
        
        gamma_skew_true = torch.randn(K, K) * 0.2
        A_true = gamma_skew_true - gamma_skew_true.T
        Gamma_true = (S_true + A_true) @ torch.linalg.inv(Omega_true)
        
    Phi_true, alpha_true = torch.randn(K, C_dim)*0.5, torch.randn(K)*0.5
    Z_true = torch.randn(D, K) - 1.0 
    Lambda_true = torch.tril(torch.ones(D, K)) * torch.exp(Z_true)
    
    subjects_data = []
    for _ in range(N):
        J_i = torch.randint(3, 6, (1,)).item()
        age_baseline = torch.rand(1) * 20 + 55
        dt = torch.rand(J_i - 1) * 3.5 + 1.5
        times = torch.cat([age_baseline, age_baseline + torch.cumsum(dt, dim=0)])
        
        t_scaled = (times - 70.0) / 10.0 
        u = torch.randn(J_i, C_dim)
        
        F_true = torch.zeros(J_i, K)
        F_true[0] = torch.randn(K) * 0.1
        
        for j in range(1, J_i):
            delta_t = times[j] - times[j-1]
            A_ij = torch.linalg.matrix_exp(-Gamma_true * delta_t)
            mu_j = (Phi_true @ u[j] + alpha_true) * t_scaled[j]
            
            Q_true = Omega_true - A_ij @ Omega_true @ A_ij.T
            Q_true = 0.5 * (Q_true + Q_true.T) + 1e-5 * torch.eye(K)
            L_Q = torch.linalg.cholesky(Q_true)
            noise = L_Q @ torch.randn(K)
            
            F_true[j] = A_ij @ F_true[j-1] + ((torch.eye(K) - A_ij) @ mu_j) + noise
            
        X_obs = F_true @ Lambda_true.T + torch.randn(J_i, D)
        subjects_data.append({'x': X_obs, 'u': u, 't': t_scaled, 'F_true': F_true})
        
    return subjects_data, {'Lambda': Lambda_true, 'Gamma': Gamma_true}

# ---------------------------------------------------------
# 4. Evaluation Wrapper
# ---------------------------------------------------------
def run_stress_test(n_runs=2):
    scenarios = [
        {"name": "1. Sparse Test",         "N": 50,  "D": 20,   "K": 3, "C": 2},
        {"name": "2. High-Dim Proteomics", "N": 100, "D": 200,  "K": 4, "C": 2},
    ]
    
    print(f"{'Scenario':<25} | {'Mode':<10} | {'Λ Corr (μ ± σ)':<15} | {'F Corr (μ ± σ)':<15} | {'Γ Corr (μ ± σ)':<15} | {'Avg Time (s)'}")
    print("-" * 105)
    
    for s in scenarios:
        for mode in ["exact", "diagonal"]:
            l_corrs, f_corrs, gamma_corrs, run_times = [], [], [], []
            
            for run_idx in range(n_runs):
                subjects_data, true_params = simulate_ad_cohort_stress(
                    s["N"], s["D"], s["K"], s["C"], theta_mode=mode, seed=300+run_idx
                )
                
                start_time = time.time()
                
                model = CLOUDS(obs_dim=s["D"], latent_dim=s["K"], covar_dim=s["C"], theta_mode=mode)
                model.pca_warm_start(subjects_data)
                smoothed_stats = model.fit_em_multistart(subjects_data, num_em_epochs=30, warmup_epochs=10, m_step_iters=15, lr=0.01)
                
                with torch.no_grad():
                    identifiable = model.get_identifiable_parameters()
                    Lambda_est = identifiable["Lambda"]
                    Gamma_est = identifiable["Gamma"]
                    mask = model.tril_mask == 1
                    
                    f_true_flat = torch.cat([subj['F_true'] for subj in subjects_data], dim=0).numpy().flatten()
                    f_est_flat = torch.cat([stat[0] for stat in smoothed_stats], dim=0).numpy().flatten()
                    
                    f_corr = np.corrcoef(f_true_flat, f_est_flat)[0, 1]
                    l_corr = np.corrcoef(true_params['Lambda'][mask].numpy(), Lambda_est[mask].numpy())[0, 1]
                    
                    if mode == "diagonal":
                        g_true = torch.diag(true_params['Gamma']).numpy()
                        g_est = torch.diag(Gamma_est).cpu().numpy()
                    else:
                        g_true = true_params['Gamma'].numpy().flatten()
                        g_est = Gamma_est.cpu().numpy().flatten()
                        
                    gamma_corr = np.corrcoef(g_true, g_est)[0, 1]
                
                elapsed = time.time() - start_time
                l_corrs.append(l_corr)
                f_corrs.append(f_corr)
                gamma_corrs.append(gamma_corr)
                run_times.append(elapsed)
            
            l_str = f"{np.mean(l_corrs):.3f} ± {np.std(l_corrs):.3f}"
            f_str = f"{np.mean(f_corrs):.3f} ± {np.std(f_corrs):.3f}"
            g_str = f"{np.mean(gamma_corrs):.3f} ± {np.std(gamma_corrs):.3f}"
            print(f"{s['name']:<25} | {mode.capitalize():<10} | {l_str:<15} | {f_str:<15} | {g_str:<15} | {np.mean(run_times):>8.1f}")
        print("-" * 105)

if __name__ == "__main__":
    run_stress_test(n_runs=2)

Scenario                  | Mode       | Λ Corr (μ ± σ)  | F Corr (μ ± σ)  | Γ Corr (μ ± σ)  | Avg Time (s)
---------------------------------------------------------------------------------------------------------
1. Sparse Test            | Exact      | 0.972 ± 0.018   | 0.914 ± 0.017   | 0.784 ± 0.121   |    137.8
1. Sparse Test            | Diagonal   | 0.923 ± 0.047   | 0.892 ± 0.029   | 0.934 ± 0.054   |    129.9
---------------------------------------------------------------------------------------------------------
2. High-Dim Proteomics    | Exact      | 0.493 ± 0.232   | 0.408 ± 0.177   | 0.613 ± 0.148   |    431.4
2. High-Dim Proteomics    | Diagonal   | 0.787 ± 0.160   | 0.642 ± 0.304   | 0.939 ± 0.054   |    400.1
---------------------------------------------------------------------------------------------------------


In [9]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import time

# ---------------------------------------------------------
# 1. Helper Function: Newton-Schulz Matrix Inversion
# ---------------------------------------------------------
def batched_newton_schulz_inverse(A, num_iters=6):
    B, N, _ = A.shape
    I = torch.eye(N, device=A.device).unsqueeze(0).expand(B, N, N)
    frob_norm_sq = torch.sum(A * A, dim=(-2, -1), keepdim=True)
    X = A.transpose(-2, -1) / (frob_norm_sq + 1e-6)
    
    for _ in range(num_iters):
        AX = torch.bmm(A, X)
        X = torch.bmm(X, (2.0 * I) - AX)
    return X

# ---------------------------------------------------------
# 2. CLOUDS Model (Scaled Priors + Staged Warmup)
# ---------------------------------------------------------
class CLOUDS(nn.Module):
    def __init__(self, obs_dim, latent_dim, covar_dim, delta=1e-4, theta_mode="exact"):
        super().__init__()
        self.D = obs_dim
        self.K = latent_dim
        self.C_dim = covar_dim
        self.delta = delta
        self.theta_mode = theta_mode
        
        # 1. Continuous-Time Dynamics (Lyapunov Parameterization)
        if self.theta_mode == "exact":
            self.L_G = nn.Parameter(torch.tril(torch.eye(self.K) + 0.1 * torch.randn(self.K, self.K)))
            self.gamma_skew = nn.Parameter(torch.randn(self.K, self.K) * 0.1)
            self.L_Omega_unc = nn.Parameter(torch.tril(torch.eye(self.K) + 0.1 * torch.randn(self.K, self.K)))
        else:
            self.log_rho = nn.Parameter(torch.randn(self.K) * 0.1 - 2.0)
            self.log_omega = nn.Parameter(torch.randn(self.K) * 0.1)
            
        # 2. Target Mean Function: mu = (Phi*u + alpha) * t
        self.Phi_int = nn.Parameter(torch.randn(self.K, self.C_dim) * 0.1)
        self.alpha_bias = nn.Parameter(torch.randn(self.K) * 0.1)
        
        # 3. Factor Loadings (Structural PCA identifiable)
        self.Z = nn.Parameter(torch.randn(self.D, self.K) - 0.5) 
        self.register_buffer('tril_mask', torch.tril(torch.ones(self.D, self.K)))
        
        # 4. Heteroscedastic Observation Noise
        self.log_psi = nn.Parameter(torch.zeros(self.D)) 

    def get_dynamics(self):
        device = self.Z.device
        if self.theta_mode == "exact":
            L_unc_tril = torch.tril(self.L_Omega_unc)
            Omega = L_unc_tril @ L_unc_tril.T + self.delta * torch.eye(self.K, device=device)
            
            G = torch.tril(self.L_G)
            S = 0.5 * (G @ G.T) + self.delta * torch.eye(self.K, device=device)
            A_skew = self.gamma_skew - self.gamma_skew.T
            
            Omega_inv = batched_newton_schulz_inverse(Omega.unsqueeze(0), num_iters=6).squeeze(0)
            Gamma = (S + A_skew) @ Omega_inv
            return Gamma, Omega, G
        else:
            Gamma = torch.diag(torch.exp(self.log_rho))
            Omega = torch.diag(torch.exp(self.log_omega))
            G = torch.sqrt(2.0 * Gamma @ Omega)
            return Gamma, Omega, G

    @torch.no_grad()
    def get_identifiable_parameters(self):
        Gamma_est, Omega_est, _ = self.get_dynamics()
        Lambda_est = self.tril_mask * torch.exp(self.Z)
        
        stds = torch.sqrt(torch.diag(Omega_est))
        D = torch.diag(stds)
        D_inv = torch.diag(1.0 / stds)
        
        Omega_corr = D_inv @ Omega_est @ D_inv
        Gamma_scaled = D_inv @ Gamma_est @ D
        Lambda_scaled = Lambda_est @ D
        Phi_scaled = D_inv @ self.Phi_int
        alpha_scaled = D_inv @ self.alpha_bias
        
        return {
            "Omega_corr": Omega_corr,
            "Gamma": Gamma_scaled,
            "Lambda": Lambda_scaled,
            "Phi": Phi_scaled,
            "alpha": alpha_scaled
        }

    def get_subject_matrices(self, Gamma, Omega, u, times):
        dt = times[1:] - times[:-1]
        device = self.Z.device
        
        Gamma_batch = Gamma.unsqueeze(0).expand(dt.shape[0], self.K, self.K)
        A_trans = torch.linalg.matrix_exp(-Gamma_batch * dt.view(-1, 1, 1))
        
        u_t, t_val = u[1:], times[1:].unsqueeze(1)
        mu = (u_t @ self.Phi_int.T + self.alpha_bias) * t_val
        
        I_batch = torch.eye(self.K, device=device).unsqueeze(0).expand(dt.shape[0], self.K, self.K)
        b_shift = torch.bmm(I_batch - A_trans, mu.unsqueeze(-1)).squeeze(-1)
        
        Omega_batch = Omega.unsqueeze(0).expand(dt.shape[0], self.K, self.K)
        A_trans_T = A_trans.transpose(1, 2)
        Q = Omega_batch - torch.bmm(A_trans, torch.bmm(Omega_batch, A_trans_T))
        Q = 0.5 * (Q + Q.transpose(1, 2)) 
            
        Lambda = self.tril_mask * torch.exp(self.Z)
        return A_trans, b_shift, dt, Lambda, Q

    def kalman_smoother(self, x_obs, A_trans, b_shift, dt, Lambda, Q):
        T = x_obs.shape[0]
        device = x_obs.device
        
        f_pred, P_pred = torch.zeros(T, self.K, device=device), torch.zeros(T, self.K, self.K, device=device)
        f_filt, P_filt = torch.zeros(T, self.K, device=device), torch.zeros(T, self.K, self.K, device=device)
        f_filt[0], P_filt[0] = torch.zeros(self.K, device=device), torch.eye(self.K, device=device)
        
        R_mat = torch.diag(torch.exp(self.log_psi))
        I_k = torch.eye(self.K, device=device)
        
        for j in range(1, T):
            idx = j - 1
            f_pred[j] = A_trans[idx] @ f_filt[j-1] + b_shift[idx]
            P_pred[j] = A_trans[idx] @ P_filt[j-1] @ A_trans[idx].T + Q[idx]
            
            if torch.isnan(x_obs[j]).all():
                f_filt[j], P_filt[j] = f_pred[j], P_pred[j]
            else:
                x_pred = Lambda @ f_pred[j]
                S_t = Lambda @ P_pred[j] @ Lambda.T + R_mat
                K_gain = P_pred[j] @ Lambda.T @ torch.linalg.inv(S_t)
                f_filt[j] = f_pred[j] + K_gain @ (x_obs[j] - x_pred)
                P_filt[j] = (I_k - K_gain @ Lambda) @ P_pred[j]
            
        f_smooth, P_smooth, P_cross = torch.zeros_like(f_filt), torch.zeros_like(P_filt), torch.zeros_like(P_filt)
        f_smooth[-1], P_smooth[-1] = f_filt[-1], P_filt[-1]
        
        for j in range(T-2, -1, -1):
            J_t = P_filt[j] @ A_trans[j].T @ torch.linalg.inv(P_pred[j+1])
            f_smooth[j] = f_filt[j] + J_t @ (f_smooth[j+1] - f_pred[j+1])
            P_smooth[j] = P_filt[j] + J_t @ (P_smooth[j+1] - P_pred[j+1]) @ J_t.T
            P_cross[j+1] = J_t @ P_smooth[j+1]
            
        return f_smooth, P_smooth, P_cross

    def expected_complete_log_posterior_vectorized(self, subjects_data, smoothed_stats, Gamma, Omega, Lambda):
        ll_obs, ll_lat = 0.0, 0.0
        
        inv_psi = torch.exp(-self.log_psi)
        L_Psi_L = Lambda.T @ torch.diag(inv_psi) @ Lambda 
        
        # Calculate total observations to scale priors correctly
        total_obs = sum([subj['x'].shape[0] for subj in subjects_data])
        
        for i, subj in enumerate(subjects_data):
            x_obs, u, times = subj['x'], subj['u'], subj['t']
            f_s, P_s, P_c = smoothed_stats[i]
            
            A_trans, b_shift, _, _, Q_exact = self.get_subject_matrices(Gamma, Omega, u, times)
            
            valid_mask = ~torch.isnan(x_obs).any(dim=1)
            if valid_mask.any():
                x_v, f_v, P_v = x_obs[valid_mask], f_s[valid_mask], P_s[valid_mask]
                trace_E = torch.sum(P_v * L_Psi_L.unsqueeze(0), dim=(1,2)) + torch.sum(f_v * (f_v @ L_Psi_L), dim=1)
                term_obs = torch.sum((x_v**2) * inv_psi, dim=1) - 2 * torch.sum(x_v * (f_v @ Lambda.T) * inv_psi, dim=1) + trace_E
                ll_obs += torch.sum(-0.5 * term_obs - 0.5 * torch.sum(self.log_psi))
                
            Q_stable = Q_exact + 1e-5 * torch.eye(self.K, device=Q_exact.device).unsqueeze(0)
            Q_inv = torch.linalg.inv(Q_stable)
            log_det_Q = torch.linalg.slogdet(Q_stable)[1]
            
            f_j, f_jm1 = f_s[1:], f_s[:-1]
            P_j, P_jm1, P_cj = P_s[1:], P_s[:-1], P_c[1:]
            
            E_jj = P_j + torch.bmm(f_j.unsqueeze(-1), f_j.unsqueeze(1))
            E_jjm1 = P_cj + torch.bmm(f_j.unsqueeze(-1), f_jm1.unsqueeze(1))
            E_jm1jm1 = P_jm1 + torch.bmm(f_jm1.unsqueeze(-1), f_jm1.unsqueeze(1))
            
            A_T = A_trans.transpose(1, 2)
            M_j = (E_jj 
                   - torch.bmm(E_jjm1, A_T) 
                   - torch.bmm(A_trans, E_jjm1.transpose(1, 2)) 
                   + torch.bmm(A_trans, torch.bmm(E_jm1jm1, A_T)))
            
            b_uns, b_T = b_shift.unsqueeze(-1), b_shift.unsqueeze(1)
            f_j_uns, f_j_T = f_j.unsqueeze(-1), f_j.unsqueeze(1)
            f_jm1_uns, f_jm1_T = f_jm1.unsqueeze(-1), f_jm1.unsqueeze(1)
            
            M_j += (- torch.bmm(f_j_uns, b_T) 
                    - torch.bmm(b_uns, f_j_T) 
                    + torch.bmm(A_trans, torch.bmm(f_jm1_uns, b_T)) 
                    + torch.bmm(b_uns, torch.bmm(f_jm1_T, A_T)) 
                    + torch.bmm(b_uns, b_T))
            
            trace_term = torch.sum(Q_inv * M_j.transpose(1, 2), dim=(1, 2))
            ll_lat += torch.sum(-0.5 * log_det_Q - 0.5 * trace_term)
            
        # Scaled Systematic Structural Priors
        log_prior_dyn = 0.0
        if self.theta_mode == "exact":
            stds = torch.sqrt(torch.diag(Omega))
            D_inv = torch.diag(1.0 / stds)
            Omega_corr = D_inv @ Omega @ D_inv
            
            # 1. LKJ Prior: Keep it gentle (eta = 1.5 instead of 2.0)
            eta = 1.5 
            # REMOVED total_obs multiplier
            log_prior_dyn += (eta - 1.0) * torch.linalg.slogdet(Omega_corr + 1e-5*torch.eye(self.K, device=Omega.device))[1]
            
            # 2. Laplace Sparsity Prior: Dramatically reduce the penalty
            lambda_laplace = 0.1 # Reduced from 1.0, and REMOVED total_obs multiplier
            log_prior_dyn -= lambda_laplace * torch.sum(torch.abs(self.gamma_skew))
            
            off_diag_G = torch.tril(self.L_G, diagonal=-1)
            log_prior_dyn -= lambda_laplace * torch.sum(torch.abs(off_diag_G))
        else:
            log_prior_dyn -= 0.5 * torch.sum(self.log_rho**2) + 0.5 * torch.sum(self.log_omega**2)

        active_Z = self.Z[self.tril_mask == 1]
        log_prior_Z = -0.5 * torch.sum(active_Z ** 2)
        log_prior_lin = -0.5 * (torch.sum(self.Phi_int**2) + torch.sum(self.alpha_bias**2))
        log_prior_psi = -0.5 * torch.sum(self.log_psi ** 2)
        
        return ll_obs + ll_lat + log_prior_dyn + log_prior_Z + log_prior_lin + log_prior_psi

    def pca_warm_start(self, subjects_data):
        with torch.no_grad():
            x_all = torch.cat([s['x'] for s in subjects_data], dim=0)
            x_valid = x_all[~torch.isnan(x_all).any(dim=1)] 
            U, S_vals, Vh = torch.linalg.svd(x_valid - x_valid.mean(dim=0), full_matrices=False)
            
            Lambda_pca = Vh[:self.K, :].T * torch.sqrt(S_vals[:self.K] / x_valid.shape[0])
            q, r = torch.linalg.qr(Lambda_pca.T)
            Lambda_tril = r.T * torch.sign(torch.diag(r.T)).unsqueeze(0)
            
            mask = self.tril_mask == 1
            self.Z.data[mask] = torch.log(torch.abs(Lambda_tril[mask]) + 1e-4)
            self.Phi_int.data.fill_(0.0); self.alpha_bias.data.fill_(0.0); self.log_psi.data.fill_(0.0)

    def fit_em_multistart(self, subjects_data, num_em_epochs=40, m_step_iters=15, lr=0.01, n_starts=5, burn_in_epochs=10):
        best_loss = float('inf')
        best_state_dict = None
        
        # Define Parameter Groups explicitly
        spatial_param_names = ['Z', 'log_psi']
        temporal_params = [p for n, p in self.named_parameters() if n not in spatial_param_names]
        spatial_params = [p for n, p in self.named_parameters() if n in spatial_param_names]
        
        print(f"  Running {n_starts} Multi-Start Burn-ins ({burn_in_epochs} epochs each)...")
        for start in range(n_starts):
            with torch.no_grad():
                if self.theta_mode == "exact":
                    nn.init.normal_(self.L_G, mean=0.0, std=0.1)
                    self.L_G.data += torch.eye(self.K, device=self.Z.device)
                    nn.init.normal_(self.gamma_skew, mean=0.0, std=0.1)
                    nn.init.normal_(self.L_Omega_unc, mean=0.0, std=0.1)
                    self.L_Omega_unc.data += torch.eye(self.K, device=self.Z.device)
                else:
                    nn.init.normal_(self.log_rho, mean=-2.0, std=0.1)
                    nn.init.normal_(self.log_omega, mean=0.0, std=0.1)
                
                nn.init.normal_(self.Phi_int, mean=0.0, std=0.1)
                nn.init.normal_(self.alpha_bias, mean=0.0, std=0.1)
            
            self.pca_warm_start(subjects_data)
            
            # Burn-in uses joint optimization just to find a rough starting basin
            opt_burn = optim.Adam(self.parameters(), lr=lr)
            start_loss = 0.0
            
            for epoch in range(burn_in_epochs):
                Gamma, Omega, _ = self.get_dynamics()
                Lambda = self.tril_mask * torch.exp(self.Z)
                
                smoothed_stats = []
                with torch.no_grad():
                    for subj in subjects_data:
                        A_trans, b_shift, dt, _, Q = self.get_subject_matrices(Gamma, Omega, subj['u'], subj['t'])
                        smoothed_stats.append(self.kalman_smoother(subj['x'], A_trans, b_shift, dt, Lambda, Q))
                
                epoch_loss = 0.0
                for m in range(m_step_iters):
                    opt_burn.zero_grad()
                    Gamma_m, Omega_m, _ = self.get_dynamics()
                    Lambda_m = self.tril_mask * torch.exp(self.Z)
                    loss = -self.expected_complete_log_posterior_vectorized(subjects_data, smoothed_stats, Gamma_m, Omega_m, Lambda_m)
                    loss.backward()
                    torch.nn.utils.clip_grad_norm_(self.parameters(), max_norm=3.0)
                    opt_burn.step()
                    epoch_loss += loss.item()
                start_loss = epoch_loss / m_step_iters
                
            if start_loss < best_loss:
                best_loss = start_loss
                best_state_dict = {k: v.clone() for k, v in self.state_dict().items()}
                
        # -------------------------------------------------------------
        # MAIN TRAINING: BLOCK COORDINATE DESCENT (Alternating M-Steps)
        # -------------------------------------------------------------
        self.load_state_dict(best_state_dict)
        print(f"  Burn-in complete. Training using Block Coordinate Descent...")
        
        opt_temporal = optim.Adam(temporal_params, lr=lr)
        opt_spatial = optim.Adam(spatial_params, lr=lr)
        
        for epoch in range(num_em_epochs - burn_in_epochs):
            # --- E-STEP ---
            Gamma, Omega, _ = self.get_dynamics()
            Lambda = self.tril_mask * torch.exp(self.Z)
            
            smoothed_stats = []
            with torch.no_grad():
                for subj in subjects_data:
                    A_trans, b_shift, dt, _, Q = self.get_subject_matrices(Gamma, Omega, subj['u'], subj['t'])
                    smoothed_stats.append(self.kalman_smoother(subj['x'], A_trans, b_shift, dt, Lambda, Q))
            
            # --- M-STEP PART 1: TEMPORAL DYNAMICS ONLY ---
            for m in range(m_step_iters):
                opt_temporal.zero_grad()
                Gamma_m, Omega_m, _ = self.get_dynamics()
                # Use detached Lambda to ensure no spatial gradients are calculated
                Lambda_detached = (self.tril_mask * torch.exp(self.Z)).detach() 
                loss = -self.expected_complete_log_posterior_vectorized(subjects_data, smoothed_stats, Gamma_m, Omega_m, Lambda_detached)
                loss.backward()
                torch.nn.utils.clip_grad_norm_(temporal_params, max_norm=3.0)
                opt_temporal.step()
                
            # --- M-STEP PART 2: SPATIAL MAPPINGS ONLY ---
            for m in range(m_step_iters):
                opt_spatial.zero_grad()
                # Use detached Dynamics to ensure no temporal gradients are calculated
                Gamma_detached, Omega_detached, _ = self.get_dynamics()
                Gamma_detached, Omega_detached = Gamma_detached.detach(), Omega_detached.detach()
                
                Lambda_m = self.tril_mask * torch.exp(self.Z)
                loss = -self.expected_complete_log_posterior_vectorized(subjects_data, smoothed_stats, Gamma_detached, Omega_detached, Lambda_m)
                loss.backward()
                # No clipping needed for spatial parameters, they are highly stable
                opt_spatial.step()
                
        return smoothed_stats

# ---------------------------------------------------------
# 3. Authentic Data Simulation Wrapper
# ---------------------------------------------------------
def simulate_ad_cohort_stress(N, D, K, C_dim, theta_mode="exact", seed=42):
    torch.manual_seed(seed)
    
    if theta_mode == "diagonal":
        rho_true = torch.linspace(0.02, 0.15, K)
        omega_true = torch.ones(K)
        Gamma_true = torch.diag(rho_true)
        Omega_true = torch.diag(omega_true)
    else:
        L_unc_true = torch.tril(torch.randn(K, K) * 0.3 + torch.eye(K))
        L_corr_true = L_unc_true / torch.norm(L_unc_true, dim=1, keepdim=True)
        Omega_true = L_corr_true @ L_corr_true.T
        
        L_G_true = torch.tril(torch.randn(K, K) * 0.3 + torch.eye(K)*0.5)
        S_true = 0.5 * (L_G_true @ L_G_true.T) + 1e-4 * torch.eye(K)
        
        gamma_skew_true = torch.randn(K, K) * 0.2
        A_true = gamma_skew_true - gamma_skew_true.T
        Gamma_true = (S_true + A_true) @ torch.linalg.inv(Omega_true)
        
    Phi_true, alpha_true = torch.randn(K, C_dim)*0.5, torch.randn(K)*0.5
    Z_true = torch.randn(D, K) - 1.0 
    Lambda_true = torch.tril(torch.ones(D, K)) * torch.exp(Z_true)
    
    subjects_data = []
    for _ in range(N):
        J_i = torch.randint(3, 6, (1,)).item()
        age_baseline = torch.rand(1) * 20 + 55
        dt = torch.rand(J_i - 1) * 3.5 + 1.5
        times = torch.cat([age_baseline, age_baseline + torch.cumsum(dt, dim=0)])
        
        t_scaled = (times - 70.0) / 10.0 
        u = torch.randn(J_i, C_dim)
        
        F_true = torch.zeros(J_i, K)
        F_true[0] = torch.randn(K) * 0.1
        
        for j in range(1, J_i):
            delta_t = times[j] - times[j-1]
            A_ij = torch.linalg.matrix_exp(-Gamma_true * delta_t)
            mu_j = (Phi_true @ u[j] + alpha_true) * t_scaled[j]
            
            Q_true = Omega_true - A_ij @ Omega_true @ A_ij.T
            Q_true = 0.5 * (Q_true + Q_true.T) + 1e-5 * torch.eye(K)
            L_Q = torch.linalg.cholesky(Q_true)
            noise = L_Q @ torch.randn(K)
            
            F_true[j] = A_ij @ F_true[j-1] + ((torch.eye(K) - A_ij) @ mu_j) + noise
            
        X_obs = F_true @ Lambda_true.T + torch.randn(J_i, D)
        subjects_data.append({'x': X_obs, 'u': u, 't': t_scaled, 'F_true': F_true})
        
    return subjects_data, {'Lambda': Lambda_true, 'Gamma': Gamma_true}

# ---------------------------------------------------------
# 4. Evaluation Wrapper
# ---------------------------------------------------------
def run_stress_test(n_runs=2):
    scenarios = [
        {"name": "1. Sparse Test",         "N": 50,  "D": 20,   "K": 3, "C": 2},
        {"name": "2. High-Dim Proteomics", "N": 100, "D": 200,  "K": 4, "C": 2},
    ]
    
    print(f"{'Scenario':<25} | {'Mode':<10} | {'Λ Corr (μ ± σ)':<15} | {'F Corr (μ ± σ)':<15} | {'Γ Corr (μ ± σ)':<15} | {'Avg Time (s)'}")
    print("-" * 105)
    
    for s in scenarios:
        for mode in ["exact", "diagonal"]:
            l_corrs, f_corrs, gamma_corrs, run_times = [], [], [], []
            
            for run_idx in range(n_runs):
                subjects_data, true_params = simulate_ad_cohort_stress(
                    s["N"], s["D"], s["K"], s["C"], theta_mode=mode, seed=300+run_idx
                )
                
                start_time = time.time()
                
                model = CLOUDS(obs_dim=s["D"], latent_dim=s["K"], covar_dim=s["C"], theta_mode=mode)
                model.pca_warm_start(subjects_data)
                smoothed_stats = model.fit_em_multistart(subjects_data, num_em_epochs=30, m_step_iters=15, lr=0.01)
                
                with torch.no_grad():
                    identifiable = model.get_identifiable_parameters()
                    Lambda_est = identifiable["Lambda"]
                    Gamma_est = identifiable["Gamma"]
                    mask = model.tril_mask == 1
                    
                    f_true_flat = torch.cat([subj['F_true'] for subj in subjects_data], dim=0).numpy().flatten()
                    f_est_flat = torch.cat([stat[0] for stat in smoothed_stats], dim=0).numpy().flatten()
                    
                    f_corr = np.corrcoef(f_true_flat, f_est_flat)[0, 1]
                    l_corr = np.corrcoef(true_params['Lambda'][mask].numpy(), Lambda_est[mask].numpy())[0, 1]
                    
                    if mode == "diagonal":
                        g_true = torch.diag(true_params['Gamma']).numpy()
                        g_est = torch.diag(Gamma_est).cpu().numpy()
                    else:
                        g_true = true_params['Gamma'].numpy().flatten()
                        g_est = Gamma_est.cpu().numpy().flatten()
                        
                    gamma_corr = np.corrcoef(g_true, g_est)[0, 1]
                
                elapsed = time.time() - start_time
                l_corrs.append(l_corr)
                f_corrs.append(f_corr)
                gamma_corrs.append(gamma_corr)
                run_times.append(elapsed)
            
            l_str = f"{np.mean(l_corrs):.3f} ± {np.std(l_corrs):.3f}"
            f_str = f"{np.mean(f_corrs):.3f} ± {np.std(f_corrs):.3f}"
            g_str = f"{np.mean(gamma_corrs):.3f} ± {np.std(gamma_corrs):.3f}"
            print(f"{s['name']:<25} | {mode.capitalize():<10} | {l_str:<15} | {f_str:<15} | {g_str:<15} | {np.mean(run_times):>8.1f}")
        print("-" * 105)

if __name__ == "__main__":
    run_stress_test(n_runs=2)

Scenario                  | Mode       | Λ Corr (μ ± σ)  | F Corr (μ ± σ)  | Γ Corr (μ ± σ)  | Avg Time (s)
---------------------------------------------------------------------------------------------------------
  Running 5 Multi-Start Burn-ins (10 epochs each)...
  Burn-in complete. Training using Block Coordinate Descent...
  Running 5 Multi-Start Burn-ins (10 epochs each)...
  Burn-in complete. Training using Block Coordinate Descent...
1. Sparse Test            | Exact      | 0.974 ± 0.016   | 0.917 ± 0.013   | 0.784 ± 0.079   |    152.3
  Running 5 Multi-Start Burn-ins (10 epochs each)...
  Burn-in complete. Training using Block Coordinate Descent...
  Running 5 Multi-Start Burn-ins (10 epochs each)...
  Burn-in complete. Training using Block Coordinate Descent...
1. Sparse Test            | Diagonal   | 0.946 ± 0.023   | 0.898 ± 0.022   | 0.924 ± 0.034   |    147.8
---------------------------------------------------------------------------------------------------------
  Runnin

In [10]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import time

# ---------------------------------------------------------
# 1. Helper Function: Newton-Schulz Matrix Inversion
# ---------------------------------------------------------
def batched_newton_schulz_inverse(A, num_iters=6):
    B, N, _ = A.shape
    I = torch.eye(N, device=A.device).unsqueeze(0).expand(B, N, N)
    frob_norm_sq = torch.sum(A * A, dim=(-2, -1), keepdim=True)
    X = A.transpose(-2, -1) / (frob_norm_sq + 1e-6)
    
    for _ in range(num_iters):
        AX = torch.bmm(A, X)
        X = torch.bmm(X, (2.0 * I) - AX)
    return X

# ---------------------------------------------------------
# 2. CLOUDS Model (Scaled Priors + Staged Warmup)
# ---------------------------------------------------------
class CLOUDS(nn.Module):
    def __init__(self, obs_dim, latent_dim, covar_dim, delta=1e-4, theta_mode="exact"):
        super().__init__()
        self.D = obs_dim
        self.K = latent_dim
        self.C_dim = covar_dim
        self.delta = delta
        self.theta_mode = theta_mode
        
        # 1. Continuous-Time Dynamics 
        if self.theta_mode == "exact":
            self.L_G = nn.Parameter(torch.tril(torch.eye(self.K) + 0.1 * torch.randn(self.K, self.K)))
            self.gamma_skew = nn.Parameter(torch.randn(self.K, self.K) * 0.1)
            self.L_Omega_unc = nn.Parameter(torch.tril(torch.eye(self.K) + 0.1 * torch.randn(self.K, self.K)))
        else:
            self.log_rho = nn.Parameter(torch.randn(self.K) * 0.1 - 2.0)
            self.log_omega = nn.Parameter(torch.randn(self.K) * 0.1)
            
        # 2. Target Mean Function
        self.Phi_int = nn.Parameter(torch.randn(self.K, self.C_dim) * 0.1)
        self.alpha_bias = nn.Parameter(torch.randn(self.K) * 0.1)
        
        # 3. Factor Loadings 
        self.Z = nn.Parameter(torch.randn(self.D, self.K) - 0.5) 
        self.register_buffer('tril_mask', torch.tril(torch.ones(self.D, self.K)))
        
        # 4. Heteroscedastic Observation Noise
        self.log_psi = nn.Parameter(torch.zeros(self.D)) 

    def get_dynamics(self):
        device = self.Z.device
        if self.theta_mode == "exact":
            L_unc_tril = torch.tril(self.L_Omega_unc)
            Omega = L_unc_tril @ L_unc_tril.T + self.delta * torch.eye(self.K, device=device)
            
            G = torch.tril(self.L_G)
            S = 0.5 * (G @ G.T) + self.delta * torch.eye(self.K, device=device)
            A_skew = self.gamma_skew - self.gamma_skew.T
            
            Omega_inv = batched_newton_schulz_inverse(Omega.unsqueeze(0), num_iters=6).squeeze(0)
            Gamma = (S + A_skew) @ Omega_inv
            return Gamma, Omega, G
        else:
            Gamma = torch.diag(torch.exp(self.log_rho))
            Omega = torch.diag(torch.exp(self.log_omega))
            G = torch.sqrt(2.0 * Gamma @ Omega)
            return Gamma, Omega, G

    @torch.no_grad()
    def get_identifiable_parameters(self):
        Gamma_est, Omega_est, _ = self.get_dynamics()
        Lambda_est = self.tril_mask * torch.exp(self.Z)
        
        stds = torch.sqrt(torch.diag(Omega_est))
        D = torch.diag(stds)
        D_inv = torch.diag(1.0 / stds)
        
        Omega_corr = D_inv @ Omega_est @ D_inv
        Gamma_scaled = D_inv @ Gamma_est @ D
        Lambda_scaled = Lambda_est @ D
        Phi_scaled = D_inv @ self.Phi_int
        alpha_scaled = D_inv @ self.alpha_bias
        
        return {
            "Omega_corr": Omega_corr,
            "Gamma": Gamma_scaled,
            "Lambda": Lambda_scaled,
            "Phi": Phi_scaled,
            "alpha": alpha_scaled
        }

    def get_subject_matrices(self, Gamma, Omega, u, times):
        dt = times[1:] - times[:-1]
        device = self.Z.device
        
        Gamma_batch = Gamma.unsqueeze(0).expand(dt.shape[0], self.K, self.K)
        A_trans = torch.linalg.matrix_exp(-Gamma_batch * dt.view(-1, 1, 1))
        
        u_t, t_val = u[1:], times[1:].unsqueeze(1)
        mu = (u_t @ self.Phi_int.T + self.alpha_bias) * t_val
        
        I_batch = torch.eye(self.K, device=device).unsqueeze(0).expand(dt.shape[0], self.K, self.K)
        b_shift = torch.bmm(I_batch - A_trans, mu.unsqueeze(-1)).squeeze(-1)
        
        Omega_batch = Omega.unsqueeze(0).expand(dt.shape[0], self.K, self.K)
        A_trans_T = A_trans.transpose(1, 2)
        Q = Omega_batch - torch.bmm(A_trans, torch.bmm(Omega_batch, A_trans_T))
        Q = 0.5 * (Q + Q.transpose(1, 2)) 
            
        Lambda = self.tril_mask * torch.exp(self.Z)
        return A_trans, b_shift, dt, Lambda, Q

    def kalman_smoother(self, x_obs, A_trans, b_shift, dt, Lambda, Q):
        T = x_obs.shape[0]
        device = x_obs.device
        
        f_pred, P_pred = torch.zeros(T, self.K, device=device), torch.zeros(T, self.K, self.K, device=device)
        f_filt, P_filt = torch.zeros(T, self.K, device=device), torch.zeros(T, self.K, self.K, device=device)
        f_filt[0], P_filt[0] = torch.zeros(self.K, device=device), torch.eye(self.K, device=device)
        
        R_mat = torch.diag(torch.exp(self.log_psi))
        I_k = torch.eye(self.K, device=device)
        
        for j in range(1, T):
            idx = j - 1
            f_pred[j] = A_trans[idx] @ f_filt[j-1] + b_shift[idx]
            P_pred[j] = A_trans[idx] @ P_filt[j-1] @ A_trans[idx].T + Q[idx]
            
            if torch.isnan(x_obs[j]).all():
                f_filt[j], P_filt[j] = f_pred[j], P_pred[j]
            else:
                x_pred = Lambda @ f_pred[j]
                S_t = Lambda @ P_pred[j] @ Lambda.T + R_mat
                K_gain = P_pred[j] @ Lambda.T @ torch.linalg.inv(S_t)
                f_filt[j] = f_pred[j] + K_gain @ (x_obs[j] - x_pred)
                P_filt[j] = (I_k - K_gain @ Lambda) @ P_pred[j]
            
        f_smooth, P_smooth, P_cross = torch.zeros_like(f_filt), torch.zeros_like(P_filt), torch.zeros_like(P_filt)
        f_smooth[-1], P_smooth[-1] = f_filt[-1], P_filt[-1]
        
        for j in range(T-2, -1, -1):
            J_t = P_filt[j] @ A_trans[j].T @ torch.linalg.inv(P_pred[j+1])
            f_smooth[j] = f_filt[j] + J_t @ (f_smooth[j+1] - f_pred[j+1])
            P_smooth[j] = P_filt[j] + J_t @ (P_smooth[j+1] - P_pred[j+1]) @ J_t.T
            P_cross[j+1] = J_t @ P_smooth[j+1]
            
        return f_smooth, P_smooth, P_cross

    def expected_complete_log_posterior_vectorized(self, subjects_data, smoothed_stats, Gamma, Omega, Lambda):
        ll_obs, ll_lat = 0.0, 0.0
        
        inv_psi = torch.exp(-self.log_psi)
        L_Psi_L = Lambda.T @ torch.diag(inv_psi) @ Lambda 
        
        for i, subj in enumerate(subjects_data):
            x_obs, u, times = subj['x'], subj['u'], subj['t']
            f_s, P_s, P_c = smoothed_stats[i]
            
            A_trans, b_shift, _, _, Q_exact = self.get_subject_matrices(Gamma, Omega, u, times)
            
            valid_mask = ~torch.isnan(x_obs).any(dim=1)
            if valid_mask.any():
                x_v, f_v, P_v = x_obs[valid_mask], f_s[valid_mask], P_s[valid_mask]
                trace_E = torch.sum(P_v * L_Psi_L.unsqueeze(0), dim=(1,2)) + torch.sum(f_v * (f_v @ L_Psi_L), dim=1)
                term_obs = torch.sum((x_v**2) * inv_psi, dim=1) - 2 * torch.sum(x_v * (f_v @ Lambda.T) * inv_psi, dim=1) + trace_E
                ll_obs += torch.sum(-0.5 * term_obs - 0.5 * torch.sum(self.log_psi))
                
            Q_stable = Q_exact + 1e-5 * torch.eye(self.K, device=Q_exact.device).unsqueeze(0)
            Q_inv = torch.linalg.inv(Q_stable)
            log_det_Q = torch.linalg.slogdet(Q_stable)[1]
            
            f_j, f_jm1 = f_s[1:], f_s[:-1]
            P_j, P_jm1, P_cj = P_s[1:], P_s[:-1], P_c[1:]
            
            E_jj = P_j + torch.bmm(f_j.unsqueeze(-1), f_j.unsqueeze(1))
            E_jjm1 = P_cj + torch.bmm(f_j.unsqueeze(-1), f_jm1.unsqueeze(1))
            E_jm1jm1 = P_jm1 + torch.bmm(f_jm1.unsqueeze(-1), f_jm1.unsqueeze(1))
            
            A_T = A_trans.transpose(1, 2)
            M_j = (E_jj 
                   - torch.bmm(E_jjm1, A_T) 
                   - torch.bmm(A_trans, E_jjm1.transpose(1, 2)) 
                   + torch.bmm(A_trans, torch.bmm(E_jm1jm1, A_T)))
            
            b_uns, b_T = b_shift.unsqueeze(-1), b_shift.unsqueeze(1)
            f_j_uns, f_j_T = f_j.unsqueeze(-1), f_j.unsqueeze(1)
            f_jm1_uns, f_jm1_T = f_jm1.unsqueeze(-1), f_jm1.unsqueeze(1)
            
            M_j += (- torch.bmm(f_j_uns, b_T) 
                    - torch.bmm(b_uns, f_j_T) 
                    + torch.bmm(A_trans, torch.bmm(f_jm1_uns, b_T)) 
                    + torch.bmm(b_uns, torch.bmm(f_jm1_T, A_T)) 
                    + torch.bmm(b_uns, b_T))
            
            trace_term = torch.sum(Q_inv * M_j.transpose(1, 2), dim=(1, 2))
            ll_lat += torch.sum(-0.5 * log_det_Q - 0.5 * trace_term)
            
        log_prior_dyn = 0.0
        if self.theta_mode == "exact":
            stds = torch.sqrt(torch.diag(Omega))
            D_inv = torch.diag(1.0 / stds)
            Omega_corr = D_inv @ Omega @ D_inv
            
            # Priors no longer scaled by total_obs since we will divide the total loss by total_obs
            eta = 1.5 
            log_prior_dyn += (eta - 1.0) * torch.linalg.slogdet(Omega_corr + 1e-5*torch.eye(self.K, device=Omega.device))[1]
            
            lambda_laplace = 0.1
            log_prior_dyn -= lambda_laplace * torch.sum(torch.abs(self.gamma_skew))
            
            off_diag_G = torch.tril(self.L_G, diagonal=-1)
            log_prior_dyn -= lambda_laplace * torch.sum(torch.abs(off_diag_G))
        else:
            log_prior_dyn -= 0.5 * torch.sum(self.log_rho**2) + 0.5 * torch.sum(self.log_omega**2)

        active_Z = self.Z[self.tril_mask == 1]
        log_prior_Z = -0.5 * torch.sum(active_Z ** 2)
        log_prior_lin = -0.5 * (torch.sum(self.Phi_int**2) + torch.sum(self.alpha_bias**2))
        log_prior_psi = -0.5 * torch.sum(self.log_psi ** 2)
        
        return ll_obs + ll_lat + log_prior_dyn + log_prior_Z + log_prior_lin + log_prior_psi

    def pca_warm_start(self, subjects_data):
        with torch.no_grad():
            x_all = torch.cat([s['x'] for s in subjects_data], dim=0)
            x_valid = x_all[~torch.isnan(x_all).any(dim=1)] 
            U, S_vals, Vh = torch.linalg.svd(x_valid - x_valid.mean(dim=0), full_matrices=False)
            
            Lambda_pca = Vh[:self.K, :].T * torch.sqrt(S_vals[:self.K] / x_valid.shape[0])
            q, r = torch.linalg.qr(Lambda_pca.T)
            Lambda_tril = r.T * torch.sign(torch.diag(r.T)).unsqueeze(0)
            
            mask = self.tril_mask == 1
            self.Z.data[mask] = torch.log(torch.abs(Lambda_tril[mask]) + 1e-4)
            self.Phi_int.data.fill_(0.0); self.alpha_bias.data.fill_(0.0); self.log_psi.data.fill_(0.0)

    def fit_em_multistart(self, subjects_data, num_em_epochs=40, warmup_epochs=15, m_step_iters=20, lr=0.01, n_starts=5, burn_in_epochs=10):
        best_loss = float('inf')
        best_state_dict = None
        
        # 1. Separate parameters explicitly
        spatial_names = ['Z', 'log_psi']
        temporal_params = [p for n, p in self.named_parameters() if n not in spatial_names]
        spatial_params = [p for n, p in self.named_parameters() if n in spatial_names]
        
        # Calculate total observations for loss normalization
        total_obs = sum([subj['x'].shape[0] for subj in subjects_data])
        
        print(f"  Running {n_starts} Multi-Start Burn-ins ({burn_in_epochs} epochs each)...")
        for start in range(n_starts):
            with torch.no_grad():
                if self.theta_mode == "exact":
                    nn.init.normal_(self.L_G, mean=0.0, std=0.1)
                    self.L_G.data += torch.eye(self.K, device=self.Z.device)
                    nn.init.normal_(self.gamma_skew, mean=0.0, std=0.1)
                    nn.init.normal_(self.L_Omega_unc, mean=0.0, std=0.1)
                    self.L_Omega_unc.data += torch.eye(self.K, device=self.Z.device)
                else:
                    nn.init.normal_(self.log_rho, mean=-2.0, std=0.1)
                    nn.init.normal_(self.log_omega, mean=0.0, std=0.1)
                
                nn.init.normal_(self.Phi_int, mean=0.0, std=0.1)
                nn.init.normal_(self.alpha_bias, mean=0.0, std=0.1)
            
            self.pca_warm_start(subjects_data)
            
            # FIX 1: STRICT TEMPORAL BURN-IN (Do NOT update spatial Z or log_psi)
            opt_burn = optim.Adam(temporal_params, lr=lr)
            start_loss = 0.0
            
            for epoch in range(burn_in_epochs):
                Gamma, Omega, _ = self.get_dynamics()
                Lambda = self.tril_mask * torch.exp(self.Z)
                
                smoothed_stats = []
                with torch.no_grad():
                    for subj in subjects_data:
                        A_trans, b_shift, dt, _, Q = self.get_subject_matrices(Gamma, Omega, subj['u'], subj['t'])
                        smoothed_stats.append(self.kalman_smoother(subj['x'], A_trans, b_shift, dt, Lambda, Q))
                
                epoch_loss = 0.0
                for m in range(m_step_iters):
                    opt_burn.zero_grad()
                    Gamma_m, Omega_m, _ = self.get_dynamics()
                    
                    # Detach Lambda completely for mathematical safety during burn-in
                    Lambda_detached = (self.tril_mask * torch.exp(self.Z)).detach()
                    
                    # FIX 2: Loss Normalization ensures stable gradients in D=200
                    loss = -self.expected_complete_log_posterior_vectorized(subjects_data, smoothed_stats, Gamma_m, Omega_m, Lambda_detached) / total_obs
                    loss.backward()
                    
                    torch.nn.utils.clip_grad_norm_(temporal_params, max_norm=2.0)
                    opt_burn.step()
                    epoch_loss += loss.item()
                start_loss = epoch_loss / m_step_iters
                
            if start_loss < best_loss:
                best_loss = start_loss
                best_state_dict = {k: v.clone() for k, v in self.state_dict().items()}
                
        # -------------------------------------------------------------
        # MAIN TRAINING: STAGED WARMUP WITH DIFFERENTIAL LEARNING RATES
        # -------------------------------------------------------------
        self.load_state_dict(best_state_dict)
        print("  Burn-in complete. Training to convergence...")
        
        opt_dynamics_only = optim.Adam(temporal_params, lr=lr)
        
        # FIX 3: Differential Learning Rates (Spatial learns 10x slower to prevent swamping)
        opt_joint = optim.Adam([
            {'params': temporal_params, 'lr': lr},
            {'params': spatial_params,  'lr': lr * 0.1} 
        ])
        
        for epoch in range(num_em_epochs - burn_in_epochs):
            Gamma, Omega, _ = self.get_dynamics()
            Lambda = self.tril_mask * torch.exp(self.Z)
            
            smoothed_stats = []
            with torch.no_grad():
                for subj in subjects_data:
                    A_trans, b_shift, dt, _, Q = self.get_subject_matrices(Gamma, Omega, subj['u'], subj['t'])
                    smoothed_stats.append(self.kalman_smoother(subj['x'], A_trans, b_shift, dt, Lambda, Q))
            
            active_opt = opt_dynamics_only if epoch < warmup_epochs else opt_joint
            
            for m in range(m_step_iters):
                active_opt.zero_grad()
                Gamma_m, Omega_m, _ = self.get_dynamics()
                Lambda_m = self.tril_mask * torch.exp(self.Z)
                
                # Protect spatial dimensions during the warmup phase
                if epoch < warmup_epochs:
                    Lambda_m = Lambda_m.detach()
                    
                loss = -self.expected_complete_log_posterior_vectorized(subjects_data, smoothed_stats, Gamma_m, Omega_m, Lambda_m) / total_obs
                loss.backward()
                
                torch.nn.utils.clip_grad_norm_(self.parameters(), max_norm=2.0)
                active_opt.step()
                
        return smoothed_stats

# ---------------------------------------------------------
# 3. Authentic Data Simulation Wrapper
# ---------------------------------------------------------
def simulate_ad_cohort_stress(N, D, K, C_dim, theta_mode="exact", seed=42):
    torch.manual_seed(seed)
    
    if theta_mode == "diagonal":
        rho_true = torch.linspace(0.02, 0.15, K)
        omega_true = torch.ones(K)
        Gamma_true = torch.diag(rho_true)
        Omega_true = torch.diag(omega_true)
    else:
        L_unc_true = torch.tril(torch.randn(K, K) * 0.3 + torch.eye(K))
        L_corr_true = L_unc_true / torch.norm(L_unc_true, dim=1, keepdim=True)
        Omega_true = L_corr_true @ L_corr_true.T
        
        L_G_true = torch.tril(torch.randn(K, K) * 0.3 + torch.eye(K)*0.5)
        S_true = 0.5 * (L_G_true @ L_G_true.T) + 1e-4 * torch.eye(K)
        
        gamma_skew_true = torch.randn(K, K) * 0.2
        A_true = gamma_skew_true - gamma_skew_true.T
        Gamma_true = (S_true + A_true) @ torch.linalg.inv(Omega_true)
        
    Phi_true, alpha_true = torch.randn(K, C_dim)*0.5, torch.randn(K)*0.5
    Z_true = torch.randn(D, K) - 1.0 
    Lambda_true = torch.tril(torch.ones(D, K)) * torch.exp(Z_true)
    
    subjects_data = []
    for _ in range(N):
        J_i = torch.randint(3, 6, (1,)).item()
        age_baseline = torch.rand(1) * 20 + 55
        dt = torch.rand(J_i - 1) * 3.5 + 1.5
        times = torch.cat([age_baseline, age_baseline + torch.cumsum(dt, dim=0)])
        
        t_scaled = (times - 70.0) / 10.0 
        u = torch.randn(J_i, C_dim)
        
        F_true = torch.zeros(J_i, K)
        F_true[0] = torch.randn(K) * 0.1
        
        for j in range(1, J_i):
            delta_t = times[j] - times[j-1]
            A_ij = torch.linalg.matrix_exp(-Gamma_true * delta_t)
            mu_j = (Phi_true @ u[j] + alpha_true) * t_scaled[j]
            
            Q_true = Omega_true - A_ij @ Omega_true @ A_ij.T
            Q_true = 0.5 * (Q_true + Q_true.T) + 1e-5 * torch.eye(K)
            L_Q = torch.linalg.cholesky(Q_true)
            noise = L_Q @ torch.randn(K)
            
            F_true[j] = A_ij @ F_true[j-1] + ((torch.eye(K) - A_ij) @ mu_j) + noise
            
        X_obs = F_true @ Lambda_true.T + torch.randn(J_i, D)
        subjects_data.append({'x': X_obs, 'u': u, 't': t_scaled, 'F_true': F_true})
        
    return subjects_data, {'Lambda': Lambda_true, 'Gamma': Gamma_true}

# ---------------------------------------------------------
# 4. Evaluation Wrapper
# ---------------------------------------------------------
def run_stress_test(n_runs=2):
    scenarios = [
        {"name": "1. Sparse Test",         "N": 50,  "D": 20,   "K": 3, "C": 2},
        {"name": "2. High-Dim Proteomics", "N": 100, "D": 200,  "K": 4, "C": 2},
    ]
    
    print(f"{'Scenario':<25} | {'Mode':<10} | {'Λ Corr (μ ± σ)':<15} | {'F Corr (μ ± σ)':<15} | {'Γ Corr (μ ± σ)':<15} | {'Avg Time (s)'}")
    print("-" * 105)
    
    for s in scenarios:
        for mode in ["exact", "diagonal"]:
            l_corrs, f_corrs, gamma_corrs, run_times = [], [], [], []
            
            for run_idx in range(n_runs):
                subjects_data, true_params = simulate_ad_cohort_stress(
                    s["N"], s["D"], s["K"], s["C"], theta_mode=mode, seed=300+run_idx
                )
                
                start_time = time.time()
                
                model = CLOUDS(obs_dim=s["D"], latent_dim=s["K"], covar_dim=s["C"], theta_mode=mode)
                model.pca_warm_start(subjects_data)
                smoothed_stats = model.fit_em_multistart(subjects_data, num_em_epochs=30, m_step_iters=15, lr=0.01)
                
                with torch.no_grad():
                    identifiable = model.get_identifiable_parameters()
                    Lambda_est = identifiable["Lambda"]
                    Gamma_est = identifiable["Gamma"]
                    mask = model.tril_mask == 1
                    
                    f_true_flat = torch.cat([subj['F_true'] for subj in subjects_data], dim=0).numpy().flatten()
                    f_est_flat = torch.cat([stat[0] for stat in smoothed_stats], dim=0).numpy().flatten()
                    
                    f_corr = np.corrcoef(f_true_flat, f_est_flat)[0, 1]
                    l_corr = np.corrcoef(true_params['Lambda'][mask].numpy(), Lambda_est[mask].numpy())[0, 1]
                    
                    if mode == "diagonal":
                        g_true = torch.diag(true_params['Gamma']).numpy()
                        g_est = torch.diag(Gamma_est).cpu().numpy()
                    else:
                        g_true = true_params['Gamma'].numpy().flatten()
                        g_est = Gamma_est.cpu().numpy().flatten()
                        
                    gamma_corr = np.corrcoef(g_true, g_est)[0, 1]
                
                elapsed = time.time() - start_time
                l_corrs.append(l_corr)
                f_corrs.append(f_corr)
                gamma_corrs.append(gamma_corr)
                run_times.append(elapsed)
            
            l_str = f"{np.mean(l_corrs):.3f} ± {np.std(l_corrs):.3f}"
            f_str = f"{np.mean(f_corrs):.3f} ± {np.std(f_corrs):.3f}"
            g_str = f"{np.mean(gamma_corrs):.3f} ± {np.std(gamma_corrs):.3f}"
            print(f"{s['name']:<25} | {mode.capitalize():<10} | {l_str:<15} | {f_str:<15} | {g_str:<15} | {np.mean(run_times):>8.1f}")
        print("-" * 105)

if __name__ == "__main__":
    run_stress_test(n_runs=2)

Scenario                  | Mode       | Λ Corr (μ ± σ)  | F Corr (μ ± σ)  | Γ Corr (μ ± σ)  | Avg Time (s)
---------------------------------------------------------------------------------------------------------
  Running 5 Multi-Start Burn-ins (10 epochs each)...
  Burn-in complete. Training to convergence...
  Running 5 Multi-Start Burn-ins (10 epochs each)...
  Burn-in complete. Training to convergence...
1. Sparse Test            | Exact      | 0.969 ± 0.012   | 0.842 ± 0.027   | 0.539 ± 0.131   |    137.0
  Running 5 Multi-Start Burn-ins (10 epochs each)...
  Burn-in complete. Training to convergence...
  Running 5 Multi-Start Burn-ins (10 epochs each)...
  Burn-in complete. Training to convergence...
1. Sparse Test            | Diagonal   | 0.764 ± 0.153   | 0.839 ± 0.075   | 0.127 ± 0.647   |    124.2
---------------------------------------------------------------------------------------------------------
  Running 5 Multi-Start Burn-ins (10 epochs each)...
  Burn-in complete.